In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from sklearn.datasets import make_circles

import imageio.v2 as imageio
from IPython.display import Markdown, display, Video
from io import BytesIO

import sys
sys.path.extend(["../../"])

from utils import bce_loss_function, brier_loss_function
from plot_utils import Arrow3D
from utils import grad_bce_loss_wrt_sigmoid_model, grad_brier_loss_wrt_sigmoid_model
from utils import computation_graph_sigmoid, computation_graph_linear, activation_function_sigmoid, create_computation_graph_linear
from utils import fit_norm2_least_square
from utils_data import generate_features
from configs import cg_rl_R01 as cg
from configs import cg_rl_R201 


$$
\newcommand{\lvec}{\mathbf{l}}
\newcommand{\xvec}{\mathbf{x}}
\newcommand{\xvect}{\mathbf{x}^T}
\newcommand{\Xmat}{\mathbf{X}}
\newcommand{\Xmatt}{\mathbf{X}^T}
\newcommand{\Amat}{\mathbf{A}}
\newcommand{\Amatt}{\mathbf{A}^T}
\newcommand{\avec}{\mathbf{a}}
\newcommand{\avect}{\mathbf{a}^T}
\newcommand{\Bmat}{\mathbf{B}}
\newcommand{\Bmatt}{\mathbf{B}^T}
\newcommand{\Cmat}{\mathbf{C}}
\newcommand{\cvec}{\mathbf{c}}
\newcommand{\Cmatt}{\mathbf{C}^T}
\newcommand{\tvec}{\mathbf{t}}
\newcommand{\tvect}{\mathbf{t}^T}
\newcommand{\Tmat}{\mathbf{T}}
\newcommand{\Tmatt}{\mathbf{T}^T}
\newcommand{\yvec}{\mathbf{y}}
\newcommand{\Ymat}{\mathbf{Y}}
\newcommand{\Ymatt}{\mathbf{Y}^T}
\newcommand{\Zmat}{\mathbf{Z}}
\newcommand{\zvec}{\mathbf{z}}
\newcommand{\wvec}{\mathbf{w}}
\newcommand{\wvect}{\mathbf{w}^T}
\newcommand{\wvecsigma}{\mathbf{w}_\sigma}
\newcommand{\wvecsigmat}{\mathbf{w}_{\sigma}^T}
\newcommand{\sigmatwovec}{\boldsymbol{\sigma^2}}
\newcommand{\Wmat}{\mathbf{W}}
\newcommand{\Wmatt}{\mathbf{W}^T}
\newcommand{\Wmatsigma}{\mathbf{W}_\sigma}
\newcommand{\Wmatsigmat}{\mathbf{W}_{\sigma}^T}
\newcommand{\Vmat}{\mathbf{V}}
\newcommand{\Vmatt}{\mathbf{V}^T}
\newcommand{\Vvec}{\mathbf{v}}
\newcommand{\Vvect}{\mathbf{v}^T}
\newcommand{\Imat}{\mathbf{I}}
\newcommand{\Sigmainv}{\Sigma^{-1}}
\newcommand{\onevec}{\mathbf{1}}
\newcommand{\onevect}{\mathbf{1}^T}
\newcommand{\dd}{\mathrm{d}}
\newcommand{\diag}{\text{diag}}
\newcommand{\pareinv}[1]{\left(#1\right)^{-1}}
\newcommand{\pare}[1]{\left(#1\right)}
\newcommand{\pareT}[1]{\left(#1\right)^{T}}
\renewcommand{\bra}[1]{\left[#1\right]}
\newcommand{\braT}[1]{\left[#1\right]^{T}}
\newcommand{\tr}[1]{\text{tr}\left(#1\right)}
\newcommand{\vvec}{\text{vec} }
\newcommand{\sgn}{\operatorname{sgn}}
$$

# Binary Classification

So far we have modelled problems in which the output is continuous on some domain. For instance, we might predict height, weight, number of euros in a sales problem etc. Targets where $t\in\mathbb{R}$ (or $\mathbb{R}^C$), depending on whether we have one or multiple outputs. These problems are usually known as regression. 

Here we deal with the problem of classification. Honestly, I do not like the distinction between regression and classification, but I shall recognize that it is quite good for people getting into this new subject. However, in the end, one realizes that classification problems are regression problems in a different domain, in particular $[0,1]$.

We start by considering the problem of binary classification, where $t$ can only take two values, $t\in\{0,1\}$. Think of a sensor telling us whether an animal is a cat ($t=0$) or a dog ($t=1$) from its weight, or whether a patient does or does not have a disease from some measurements. Other classification algorithms consider the labels $t\in\{-1,1\}$, for computational reasons (if I do not remember bad). 

## One dimensional binary classification: $f:\mathbb{R}\rightarrow[0,1]$

One-dimensional binary classification stands for problems where, for an input $x\in\mathbb{R}$, we want to predict a label $t\in\{0,1\}$. As in the [regression theory](1_Regression_Shallow_theory.ipynb), we will keep working with a toy example we can fully visualize: $x$ is the weight of an animal, and $t$ tells us whether it is a dog ($t=1$) or a cat ($t=0$). Here are our seven observations:

$$
\begin{split}
(x_1,t_1) &= (-0.13459237,0)\\
(x_2,t_2) &= (-3.3015387,0)\\
(x_3,t_3) &= (0.74481176,0)\\
(x_4,t_4) &= (2.62434536,1)\\
(x_5,t_5) &= (0.38824359,1)\\
(x_6,t_6) &= (0.47182825,1)\\
(x_7,t_7) &= (-0.07296862,1)\\
\end{split}
$$

Let's plot these values.


In [ ]:
color_c0 = 'C0'
color_c1 = 'C1'

x_data = np.array([-0.13459237,-3.3015387,0.74481176,2.62434536,0.38824359,0.47182825,-0.07296862]).reshape(7,1)
t_data = np.array([0,0,0,1,1,1,1]).reshape(7,1)

idx_class0 = t_data == 0
idx_class1 = t_data == 1

fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.plot(x_data[idx_class0], t_data[idx_class0], 'o', color=color_c0, markersize=8, label='class 0 (cat)')
ax.plot(x_data[idx_class1], t_data[idx_class1], '*', color=color_c1, markersize=10, label='class 1 (dog)')
ax.set_xlabel(cg.data_x_name)
ax.set_ylabel('t')
ax.set_xlim([cg.data_x_lim_l, cg.data_x_lim_u])
ax.set_ylim([-0.3, 1.3])
ax.legend()


### A model for this data

The linear model we have used so far, $y=wx+b$, does not make sense for this problem: our target $t$ is not an arbitrary real number; it is a label that can only take the values $0$ or $1$. 

The question now is, are we really interested in just predicting the label 0 or the label 1?. The answer is no; what we are really interested in is predicting the probability that an input takes the value $t=1$. Why? Well, do you think it is the same to say a patient you have cancer with 0.9 probability as with 0.2? Or, said in another way, if a doctor says you have cancer with 20% probability, would you go home and never come back to the doctor?

Obviously, it depends on the problem. At some point, we might only be interested in making a classification decision, ie $t=0$ or $t=1$. But that's fine; if we manage to get a prediction which says the probability of event $t = 1$ is $0.6$, we might decide to classify everything as $t=1$ when the probability exceeds 0.5. In a more critical scenario (cancer one), we might decide to classify "no cancer" only when the probability is above $0.95$. So the classification scenario deals with probability assignments to inform decisions (the regression one as well, but it is not so intuitive).

Thus, labels $t=0$ and $t=1$ represent, in reality, the probability of $x$ belonging to any of the two classes. So, for instance, in the cat and dog problem we can label $t=1$ dog and $t=0$ cat. Thus, if $x$ is a cat, labeled with $t=0$, it is because the probability of being a dog is 0. 

Following the [Generalized Linear Models](4_Generalized_Linear_Models_theory.ipynb) chapter, the fix is to introduce a link function $\Phi$ that maps the unrestricted linear predictor $z=wx+b$ into the range we actually need here, $[0,1]$. There are several candidate links with that range:

$$
\sigma(z)=\frac{1}{1+e^{-z}} \quad\text{(sigmoid)}, \qquad \Phi_{\text{probit}}(z)=\int_{-\infty}^z\mathcal{N}(u;0,1)\,\dd u \quad\text{(probit)}, \qquad \mathbb{1}[z>0] \quad\text{(Heaviside)}
$$

We can discard the Heaviside right away: it is a step function, with zero derivative everywhere it is defined (and undefined exactly at $z=0$), so no gradient-based method could ever be used to fit a model built on top of it. This hard threshold idea is classically used by the perceptron algorithm. This algorithm is not based on gradient descent, although if I do not remember badly, I once found  a blog or something connecting this algorithm to the underlying loss and activation function whose gradient results in the perceptron algorithm. Well, for our purposes, where we mostly need this gradient, it does not make sense. Also, since we want a probability, it is useless at all.

That leaves the sigmoid and the probit, which have essentially the same shape (both map $\mathbb{R}\to [0,1]$ monotonically, both symmetric around $z=0$) and behave almost identically in practice. We choose the sigmoid for several reasons:

* Canonical link of the underlying probability distribution (Bernoulli), making the resulting loss gradient particularly simple.
* It is cheap to compute. A single exponential; it as closed-form inverse. The probit link, in contrast, is the Gaussian CDF, which has no elementary closed form and must be evaluated numerically (via the error function).
* Its coefficients have a direct, popular interpretation: exponentiating a logistic regression coefficient gives an odds ratio, a quantity widely used and reported in applied statistics (epidemiology, social sciences, etc.). Probit coefficients live in "Gaussian z-score" units instead, which are harder to communicate.
* Empirically, it makes very little difference which of the two we pick: the two curves are nearly indistinguishable in shape (the logistic has slightly heavier tails than the Gaussian), so the choice rarely changes predictive performance.

None of this makes the probit link wrong, though; it is simply more convenient elsewhere. Its main advantage is analytical: convolving the Gaussian CDF with a Gaussian density has a closed form,

$$
\int \Phi_{\text{probit}}(z)\,\mathcal{N}(z;\mu,\sigma^2)\,\dd z = \Phi_{\text{probit}}\pare{\frac{\mu}{\sqrt{1+\sigma^2}}},
$$

while the analogous integral against the sigmoid does not. This is exactly why the probit, rather than the sigmoid, is the workhorse link whenever we need to marginalize a Gaussian-distributed linear predictor through the link, e.g. in Gaussian Process classification or Bayesian probit models.

Although both the sigmoid and probit can be used to parameterize a Bernoulli distribution, the probit has an alternative interpretation in terms of the statistical model being implemented, which seems to be popular in econometrics: https://en.wikipedia.org/wiki/Probit_model


### Displaying models

If we use $\wvec=[w,b]$ as we do throughout the book, then our new model is:

$$
\begin{split}
z = \xvect\wvec\\
y=\frac{1}{1+e^{-z}}
\end{split}
$$

where $y\in(0,1)$ is interpreted as the model's predicted probability that $t=1$ (dog), given $x$.  In other words: $y=p(t=1\mid\xvec)$. From this, clearly $p(t=0\mid\xvec) = 1-p(t=1\mid\xvec)$ because probabilities must sum one.


As with the linear model, there are infinitely many possible values for $w,b$. Let's plot three candidate models.


In [ ]:
np.random.seed(cg.seed)

x_range = np.linspace(cg.data_x_range_l, cg.data_x_range_u, cg.N_domain_x).reshape(-1, 1)

fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.plot(x_data[idx_class0], t_data[idx_class0], 'o', color=color_c0, markersize=8, label='class 0 (cat)')
ax.plot(x_data[idx_class1], t_data[idx_class1], '*', color=color_c1, markersize=10, label='class 1 (dog)')

for i in range(3):
    w, b = create_computation_graph_linear(1, 1)
    y_range = computation_graph_sigmoid(x_range, w, b)
    y_data = computation_graph_sigmoid(x_data, w, b)
    if i == 0:
        ax.plot(x_range, y_range, color=f"C{i+2}", label='candidate model (all domain)')
        ax.plot(x_data, y_data, '*', markersize = 5,  color=f"C{i+2}", label='predictions at the data')
    else:
        ax.plot(x_range, y_range, color=f"C{i+2}")
        ax.plot(x_data, y_data,'*', markersize = 5,  color=f"C{i+2}")

ax.set_xlabel(cg.data_x_name)
ax.set_ylabel(cg.data_y_name)
ax.set_xlim([cg.data_x_lim_l, cg.data_x_lim_u])
ax.set_ylim([cg.data_y_lim_l, cg.data_y_lim_u])
ax.legend(loc='center left')


### Loss functions

We now have a model, $y=\sigma(z)$, producing a predicted probability that $t=1$. What is left is to define a loss $L(t,y)$ that measures how good this prediction is, so that we can then fit $\wvec$ by minimizing it over our data.

A natural candidate is the **binary cross-entropy** (BCE), also known as the *log-loss*:


#### Binary Cross Entropy

The Binary Cross Entropy (BCE) loss is given by:

$$
L_{\text{BCE}} = -\sum_{n=1}^N \bra{t^{(n)}\log y^{(n)} + (1-t^{(n)})\log(1-y^{(n)})}
$$

Let's see, intuitively, why the log loss is a good candidate model and what is expected behaviour. Remember a loss function measures how good a candidate model is (on expectation) to represent the data. Thus, we need a loss that penalizes wrong predictions (ie assigning $y=0$ when the label says $t=1$ and vice versa), and that has a loss of zero when we do things correctly. Let's visualize this loss function.

In [ ]:
y_grid = np.linspace(1e-3, 1 - 1e-3, 400).reshape(-1, 1)

loss_t1 = bce_loss_function(np.ones_like(y_grid), y_grid)
loss_t0 = bce_loss_function(np.zeros_like(y_grid), y_grid)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(y_grid, loss_t1, linewidth = 4, color=color_c1, label=r"$t=1$: $-\log y$")
ax.plot(y_grid, loss_t0, linewidth = 4, color=color_c0, label=r"$t=0$: $-\log(1-y)$")
ax.set_xlabel("$y$ (predicted probability)")
ax.set_ylabel(r"$L_{\text{BCE}}(t,y)$")
ax.set_ylim(0, 8)
ax.set_title("Binary cross-entropy loss vs. predicted probability")
ax.legend()
ax.grid(True)


The plot makes clear what we asked for: a loss that penalizes wrong predictions and vanishes when we get things right.

* Each curve touches $0$ exactly at the correct extreme: $-\log y\to0$ as $y\to1$ (when $t=1$), and $-\log(1-y)\to0$ as $y\to0$ (when $t=0$). The loss is zero only when the model is fully, correctly confident, not merely "close enough": a prediction of $y=0.9$ for $t=1$ still pays a (small) price.
* Each curve blows up, without bound, at the wrong extreme: $-\log y\to\infty$ as $y\to0$ when $t=1$, and symmetrically for $t=0$. A confidently wrong prediction is punished arbitrarily harder than a merely uncertain one ($y$ close to $0.5$).
* Both curves are smooth and monotonic over the whole of $(0,1)$: no flat regions, no kinks, and no ambiguity about which direction improves the prediction.

This combination, zero only at perfect confidence, unbounded penalty for confident mistakes, and smooth everywhere in between, is exactly what we want from a training signal: a model that is very wrong should feel a very strong correction, and a model that is exactly right should feel none at all. This is what justifies the binary cross-entropy as an adequate cost function for this problem.


#### Brier Loss

BCE is not the only sensible way to score a probabilistic prediction $y$ against a binary outcome $t$. Another one, which you have actually already met, is to just reuse the squared loss on $y$ directly:

$$
L_{\text{Brier}} = \sum^N_{n=1} (t^{(n)}-y^{(n)})^2
$$

i.e., the sum of squared errors between the predicted probability and the observed label. In this context, where $y$ is a probability and $t$ a binary outcome rather than an arbitrary real-valued target, this is known as the **Brier score**, and it is an example of what is called a *scoring rule*: a loss function specifically designed to score probabilistic predictions.

#### Discussion on both losses.

Both losses are in fact instances of a more general recipe, generalized entropies induced by different Bregman divergences, but that generality is not needed here; we only need  that both are valid, honest ways to score our sigmoid's output. These divergences induce different types of scoring rules.

The difference, perhaps, in these two losses relies on:

* The BCE loss plus a linear sigmoid model results in a convex loss function. So there is only one minimum. This does not happen with the squared loss with a sigmoid model. In this case, and in contrast to what we have seen, there is more than one minimum.
* The BCE loss penalizes very wrong predictions more (with infinite loss). The Brier loss only penalizes with one.
* The BCE loss models the data through a Bernoulli likelihood, which directly targets probability distributions over discrete binary outcomes. The Brier loss implicitly assumes data is Gaussian distributed (as we know), which is misspecified for this kind of problem. Why? Because Gaussian distributions have support over all the reals, while we are interested only in predicting within a plausible probabilistic range.
* Since the BCE loss is a scoring rule, this means that, in fact, we can target problems in which $t$ is not directly $0$ or $1$, but any value between 0  and 1. This includes problems in which $x$, by construction, only represents a small amount of probability of belonging to one of the two events. The Bernoulli likelihood does not allow modeling this (it only models outcomes 1 or 0). So the scoring rule perspective of the BCE is more general. In fact, in my PhD thesis slides I derived all this. So need to place it here at some point. This also happens with the Brier loss.

We now illustrate both losses and show how one of them is convex, and the other is not. As an **exercise** show the connection of the BCE loss with the Bernoulli likelihood. First, let's see how the losses are assigned to a couple of models.

Another **Exercsise** is to show that the BCE loss is convex both when we fix $b$ and for $b$ and $w$. In this last case we can make use of the determinant of the Hessian.

In [ ]:
## fix seed so that randomness is controlled.
np.random.seed(cg.seed)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

# domain over where we want to plot the function implemented by the NNet
x_range = np.linspace(cg.data_x_range_l,cg.data_x_range_u, N_points_domain).reshape((N_points_domain,1))

## get number of losses to display
num_losses = len(cg.losses)

## display data again
fig, ax_list = plt.subplots(1,num_losses, figsize=(20,5))
if num_losses == 1:
    ax_list = [ax_list]

idx_class0 = t_data == 0
idx_class1 = t_data == 1

## annotating all 7 points clutters the plot; keep only the two extreme points
label_idx = [x_data.argmin(), x_data.argmax()]

for loss_itet,ax in enumerate(ax_list):

    ax.plot(x_data[idx_class0],t_data[idx_class0],'o', color = color_c0, markersize = 8, zorder = 3, label = 'data observations class 0 cat')
    ax.plot(x_data[idx_class1],t_data[idx_class1],'*', color = color_c1,markersize = 8, zorder = 3, label = 'data observations class 1 dog')

    ax.set_xlabel(cg.data_x_name)
    ax.set_ylabel(cg.data_y_name)
    ax.set_title(cg.losses[loss_itet].loss_name)
    ax.set_ylim([cg.losses[loss_itet].loss_y_lim_l,cg.losses[loss_itet].loss_y_lim_u])
    ax.set_xlim([cg.losses[loss_itet].loss_x_lim_l,cg.losses[loss_itet].loss_x_lim_u])    

## =========================================================================================
## Create several possible functions that our specific neural network can implement and plot

# to save parameters to use later
w_save = []
b_save = []

for i, model_id in zip(range(3),['a','b','c']):

    # initialize one of our networks
    w, b = create_computation_graph_linear(1,1)

    # projection from input x to output y through computational graph
    y_range = computation_graph_sigmoid(x_range,w, b)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_data = computation_graph_sigmoid(x_data,w,b)
    
    # for each of the losses compute the loss and display:
    for loss_itet,ax in enumerate(ax_list):
        loss = cg.losses[loss_itet].loss_fun(t_data, y_data)
        
        if i == 0:
            ax.plot(x_data, y_data,'*', markersize = 5, color = f"C{i+1}", zorder = 3, label = 'Predictions at training input data')
            ax.plot(x_range,y_range, color = f"C{i+1}", zorder = 1, label = 'function on all the domain' )

            for j in label_idx:
                ax.text(x_data[j], y_data[j] - cg.losses[loss_itet].total_loss_display_inc, f"$d(y_{j+1},t_{j+1})={float(loss[j]):.2}$", fontsize=12, ha="center", color = f"C{i+1}") 

            ax.text(1,-cg.losses[loss_itet].total_loss_display_inc*i, f"$L(w_{model_id},b_{model_id}) = {float(np.sum(loss)):.2f}$", color = f"C{i+1}")    

        else:
            ax.plot(x_data, y_data,'*', markersize = 5,  color = f"C{i+1}", zorder = 3)
            ax.plot(x_range,y_range, color = f"C{i+1}", zorder = 1)

            for j in label_idx:
                ax.text(x_data[j], y_data[j] + j*cg.losses[loss_itet].total_loss_display_inc, f"$d(y_{j+1},t_{j+1})={float(loss[j]):.2}$", fontsize=12, ha="center", color = f"C{i+1}")

            ax.text(1,-cg.losses[loss_itet].total_loss_display_inc*i, f"$L(w_{model_id},b_{model_id}) = {float(np.sum(loss)):.2f}$", color = f"C{i+1}")    
            



### Visualizing Loss Functions

Exactly as with the squared and absolute losses, the loss is a function of the parameters $L(w,b,\Xmat,\tvec)$, and we can visualize it as such. As done previously, let's fix $b=0.5$ and plot both losses as a function of $w$ alone.


In [ ]:
## ============================================================================== ##
## display loss as a function of weight parameter (loss incurred by each network) ##
## ============================================================================== ##
## Let's see the associated loss to each possible function but seeing the loss
## as a function of the weight parameter. To do so we fix the bias value.
## We show two different losses: squared (top) and absolute ( bottom )
## I repeat code from above but computing and plotting the loss.


## ======================== ##
## Simulation configuration ##
## ======================== ##
fixed_bias = cg.fixed_bias

## fix seed so that randomness is controlled.
np.random.seed(cg.seed)

## for data plotting
idx_class0 = t_data == 0
idx_class1 = t_data == 1

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

## get number of losses to display
num_losses = len(cg.losses)

## ================ ##
## For plot display ##
## ================ ##
## create figure box
fig, ax_list = plt.subplots(num_losses,2, figsize = (10,10))
# wrap into list of list for this case since when num_losses is 1 it is not wrapped.
if num_losses == 1:
    ax_list = [ax_list]

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

## =============================================
## Specify input output to the computation graph
D_in = 1
D_out = 1

## ================================================================================
## Create several possible functions that our specific neural network can implement

# to save individual losses, expected losses and parameters used
loss_acc = [[] for i in range(num_losses)]
expected_loss_acc = [[] for i in range(num_losses)]

# domain over where we want to plot the function implemented by the NNet
x_range = np.linspace(cg.data_x_range_l,cg.data_x_range_u, N_points_domain).reshape((N_points_domain,1))

w_acc = []
w_range = []
# Compute the loss function over 100 possible neural net.
for i in range(cg.N_models_simulation):
    
    # initialize one of our networks
    w, b = create_computation_graph_linear(1,1, std = 4)

    # projection from input x to output y through computational graph
    y_range = computation_graph_sigmoid(x_range,w, fixed_bias)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_data = computation_graph_sigmoid(x_data,w, fixed_bias)

    # for each of the losses:
    for loss_itet,ax in enumerate(ax_list):
        
        # compute the loss at the predictions
        loss = cg.losses[loss_itet].loss_fun(t_data, y_data)
    
        # accumulate the loss and save both individual and accumulated losses
        loss_acc[loss_itet].append(loss)
        expected_loss_acc[loss_itet].append(np.sum(loss))
        
    # save parameter used to compute the loss
    w_range.append(np.squeeze(w))
    w_acc.append(w)

# sort loss and weights to interactive plot later
idx = np.argsort(w_range)
sorted_expected_loss_acc = []
for _ in expected_loss_acc:
    sorted_expected_loss_acc.append(np.array(_)[idx])
## ===================================================================== ##
## Display different models sequentially, alongside its loss, in a video ##

# Display the loss over the possible plaussible functions.
w_old = []
expected_loss_old = [ [] for _ in range(num_losses)]

for i,w in zip(range(cg.N_models_display),w_acc):  
    
    ## repeat the projection from input x to output y through computational graph
    y_range = computation_graph_sigmoid(x_range,w, fixed_bias)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_data = computation_graph_sigmoid(x_data,w, fixed_bias)
    
    ## for subsequent plotting
    w = np.squeeze(w)
    
    ## ============== ##
    ## Start plotting ##
    ## ============== ##
    
    ## clean up points
    for loss_itet in range(num_losses):
        for ax in ax_list[loss_itet]:
            ax.cla()

    ## display data and loss function
    for loss_itet in range(num_losses):
        
        ## plot data
        ax_1 = ax_list[loss_itet][0]
        ax_1.plot(x_data[idx_class0],t_data[idx_class0],'o', color = color_c0, markersize = 8, label = 'data observations class 0 cat')
        ax_1.plot(x_data[idx_class1],t_data[idx_class1],'*', color = color_c1,markersize = 8, label = 'data observations class 1 dog')
        ax_1.set_xlabel(cg.data_x_name)
        ax_1.set_ylabel(cg.data_y_name)
        ax_1.set_title(cg.losses[loss_itet].loss_name)
        ax_1.set_ylim([cg.losses[loss_itet].loss_y_lim_l,cg.losses[loss_itet].loss_y_lim_u])
        ax_1.set_xlim([cg.losses[loss_itet].loss_x_lim_l,cg.losses[loss_itet].loss_x_lim_u])
        
        ## plot loss function
        ax_2 = ax_list[loss_itet][1]
        ax_2.plot(np.array(w_range)[idx], sorted_expected_loss_acc[loss_itet] )
        ax_2.set_xlabel('weight values')
        ax_2.set_ylabel(cg.losses[loss_itet].loss_name)
        
        ## Plot function
        ax_1.plot(x_data[idx_class0], y_data[idx_class0],'x', markersize = 5, color = f"C0", label = 'Predictions at training input data')
        ax_1.plot(x_data[idx_class1], y_data[idx_class1],'x', markersize = 5, color = f"C1")
        ax_1.plot(x_range,y_range, color = f"C2", label = 'function on all the domain' )

        ## Plot individual losses
        for j in range(len(x_data)):
            ax_1.text(x_data[j], j*cg.losses[loss_itet].per_point_loss_display_y_inc, f"$d(y_{j+1},t_{j+1})={float(loss_acc[loss_itet][i][j]):.2}$", fontsize=8, ha="center", color = f"C1") 
    
        ## Plot expected loss
        ax_1.text(cg.losses[loss_itet].total_loss_display_x,cg.losses[loss_itet].total_loss_display_y, f"$L(w = {w:.2f},b = {fixed_bias}) = {expected_loss_acc[loss_itet][i]:.2f}$", color = f"C1")  
        ax_1.text(cg.losses[loss_itet].total_loss_display_x,cg.losses[loss_itet].total_loss_display_y-2*cg.losses[loss_itet].total_loss_display_inc, f"$y = {w:.2f} \cdot x + {fixed_bias}$", color = f"C1")  
    
        ## Plot already displayed losses
        ax_2.plot(w_old, expected_loss_old[loss_itet], '*', color = 'C0')

        ## Plot the loss in the loss function view
        ax_2.plot(w, expected_loss_acc[loss_itet][i], '*', color = f"C1")
        ax_2.text(w, expected_loss_acc[loss_itet][i]+cg.losses[loss_itet].total_loss_display_loss_plot_inc, f"$L(w = {w:.2f},b = {fixed_bias}) = {expected_loss_acc[loss_itet][i]:.2f}$", color = f"C1")    
   
        # save old loss to display in next figure iteration
        expected_loss_old[loss_itet].append(expected_loss_acc[loss_itet][i])

    # save old to display in next figure
    w_old.append(w)

    ## Cortesía de chatGPT (desde linea siguiente hasta el final de esta celda):
    ## save images for later display
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)  

writer.close() 
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

We clearly observe how the BCE loss results in a convex function, while the Brier loss is non-convex, although it has a unique minimum. Let's now check this loss on  a 2 dimensional, as a function of $w$ and $b$. 


In [ ]:
## ======================================================================================= ##
## display loss as a function of weight and bias parameter (loss incurred by each network) ##
## ======================================================================================= ##
## Let's see the associated loss to each possible function but seeing the loss
## as a function of the weight and bias parameter. 
## We show two different losses: squared (top) and absolute ( bottom )

## fix seed so that randomness is controlled.
np.random.seed(cg.seed)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

## get number of losses to display
num_losses = len(cg.losses)

## for data plotting
idx_class0 = t_data == 0
idx_class1 = t_data == 1

## ================ ##
## For plot display ##
## ================ ##
## create figure box
fig = plt.figure(figsize = (10,8))
ax_list = []
counter = 1
for i in range(num_losses):
    ax_1 = fig.add_subplot(num_losses,2,counter)
    ax_2 = fig.add_subplot(num_losses,2,counter + 1, projection='3d')
    ax_2.view_init(elev=30, azim=cg.losses[i].full_loss_display_azim)
    
    counter += 2
    
    ax_list.append([ax_1,ax_2])

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

## ===========================================
## Neural network specification for each layer

# neurons of input layer
n_in = 1
# neurons of output layer
n_out = 1

## ===================================================================================================
## Create several possible functions that our specific neural network can implement and compute losses
'''
**optimal parameters for bce loss (global)**
weight = 1.1246184
bias = 0.12033531


**optimal parameters for brier loss (local)**
weight = 0.85201174
bias = 0.2264343
'''

expected_loss_mesh = []
for loss_itet in range(num_losses):

    ## To do so we need a mesh
    w_mesh, b_mesh = np.meshgrid(
        np.linspace(cg.losses[loss_itet].w_range_l_2d,cg.losses[loss_itet].w_range_u_2d,cg.N_models_simulation),
        np.linspace(cg.losses[loss_itet].b_range_l_2d,cg.losses[loss_itet].b_range_u_2d,cg.N_models_simulation)
    )

    # reshape x_data and t_data for computations. t_data uses broadcasting
    x_data_expanded = x_data[:,np.newaxis]
    t_data_expanded = t_data[:,np.newaxis]
    
    # compute linear projection at all pairs of points
    y_data_expanded = activation_function_sigmoid( w_mesh * x_data_expanded + b_mesh) 
    
    if cg.losses[loss_itet].loss_name == 'Binary Cross Entropy':
        # add new axis for correct indexation in the function
        t_data_expanded = np.squeeze(t_data_expanded)

    # compute loss
    _expected_loss = np.sum(cg.losses[loss_itet].loss_fun(t_data_expanded, y_data_expanded), axis = 0)
    
    ## saturate to maximum value for display
    max_val = np.max(_expected_loss[_expected_loss != np.inf ])
    min_val = np.min(_expected_loss[_expected_loss != -np.inf ])
    _expected_loss[_expected_loss == np.inf ] = max_val
    _expected_loss[_expected_loss == -np.inf ] = min_val
    
    expected_loss_mesh.append(_expected_loss)

# to save individual losses, expected losses and parameters used
loss_acc = [[] for _ in range(num_losses)]
expected_loss_acc = [[] for _ in range(num_losses)]
    
# domain over where we want to plot the function implemented by the NNet
x_range = np.linspace(cg.data_x_range_l,cg.data_x_range_u, N_points_domain).reshape((N_points_domain,1))
    
w_acc = []
w_range = []
b_acc = []
b_range = []

# Compute the loss funciton over some possible models.
for i in range(cg.N_models_display):

    # initialize one of our networks
    w, b = create_computation_graph_linear(n_in,n_out)

    # projection from input x to output y through computational graph
    y_range = computation_graph_sigmoid(x_range,w,b)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_data = computation_graph_sigmoid(x_data,w,b)

    for loss_itet in range(num_losses):
        
        # compute loss at the predictions
        loss = cg.losses[loss_itet].loss_fun(t_data, y_data)
        
        # acumulate loss and parameter used
        loss_acc[loss_itet].append(loss)
        expected_loss_acc[loss_itet].append(np.sum(loss))
        
    w_range.append(np.squeeze(w))
    b_range.append(np.squeeze(b))
    w_acc.append(w)
    b_acc.append(b)

## =========================================================
## Display different models sequentially, alongside its loss.
#  The first 3 ones are the ones generated in step 1.2

# variables to keep track of old weight losses display to show the overall loss function.
w_old = []
b_old = []
expected_loss_old = [ [] for _ in range(num_losses)]

for i,w,b in zip(range(len(w_acc)),w_acc,b_acc):  
    
    if i == 0 or i == 1 or i == 2:
        color = f"C{i+1}"
    else:
        color = 'C5'
    
    ## repeat the projection from input x to output y through computational graph
    y_range = computation_graph_sigmoid(x_range,w,b)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_data = computation_graph_sigmoid(x_data,w,b)
    
    ## for subsequent plotting
    w = np.squeeze(w)
    b = np.squeeze(b)
    
    ## clean up points
    for loss_itet in range(num_losses):
        for ax in ax_list[loss_itet]:
            ax.cla()

    ## display data and loss function
    for loss_itet in range(num_losses):
        ax_1 = ax_list[loss_itet][0]
        ax_2 = ax_list[loss_itet][1]
        
        ## plot data
        ax_1 = ax_list[loss_itet][0]
        ax_1.plot(x_data[idx_class0],t_data[idx_class0],'o', color = color_c0, markersize = 8, label = 'data observations class 0 cat')
        ax_1.plot(x_data[idx_class1],t_data[idx_class1],'*', color = color_c1,markersize = 8, label = 'data observations class 1 dog')
        ax_1.set_xlabel(cg.data_x_name)
        ax_1.set_ylabel(cg.data_y_name)
        ax_1.set_title(cg.losses[loss_itet].loss_name)
        ax_1.set_ylim([cg.losses[loss_itet].loss_y_lim_l,cg.losses[loss_itet].loss_y_lim_u])
        ax_1.set_xlim([cg.losses[loss_itet].loss_x_lim_l,cg.losses[loss_itet].loss_x_lim_u])
        
        ## display loss functions
        ax_2.plot_surface(w_mesh, b_mesh, expected_loss_mesh[loss_itet], cmap = 'viridis', alpha = 0.75 )
        ax_2.set_xlabel('weight values')
        ax_2.set_ylabel('bias values')
        ax_2.set_zlabel(cg.losses[loss_itet].loss_name)

        ## Plot function
        ax_1.plot(x_data[idx_class0], y_data[idx_class0],'x', markersize = 5, color = f"C0", label = 'Predictions at training input data')
        ax_1.plot(x_data[idx_class1], y_data[idx_class1],'x', markersize = 5, color = f"C1")
        ax_1.plot(x_range,y_range, color = f"C2", label = 'function on all the domain' )

        ## Plot individual losses
        for j in range(len(x_data)):
            ax_1.text(x_data[j], j*cg.losses[loss_itet].per_point_loss_display_y_inc, f"$d(y_{j+1},t_{j+1})={float(loss_acc[loss_itet][i][j]):.2}$", fontsize=8, ha="center", color = color) 

        ## Plot expected loss
        ax_1.text(cg.losses[loss_itet].total_loss_display_x,cg.losses[loss_itet].total_loss_display_y, f"$L(w = {w:.2f},b = {b:.2f})= {expected_loss_acc[loss_itet][i]:.2f}$", color = color)  
        ax_1.text(cg.losses[loss_itet].total_loss_display_x,cg.losses[loss_itet].total_loss_display_y-2*cg.losses[loss_itet].total_loss_display_inc, f"$y = {w:.2f} \cdot x + {b:.2f}$", color = color)  

        ## Plot already displayed losses
        ax_2.plot(w_old, b_old, expected_loss_old[loss_itet], '*', color = 'C0')
        
        ## Plot the loss in the loss function view
        ax_2.plot(w, b, expected_loss_acc[loss_itet][i], '*', color = color)
        ax_2.text(w,b,expected_loss_acc[loss_itet][i]+cg.losses[loss_itet].total_loss_display_loss_plot_inc, f"$L(w = {w:.2f},b = {b:.2f}) = {expected_loss_acc[loss_itet][i]:.2f}$", color = color, zorder = 2, ha = 'center')       

        # save old to display in next figure. first 3 models are not added to the old list to not be plotted as old and so that we can highlight them
        # as required.
        expected_loss_old[loss_itet].append(expected_loss_acc[loss_itet][i])
    
    # first 3 models are not added to the old list to not be plotted as old and so that we can highlight them
    # as required.
    w_old.append(w)
    b_old.append(b)
    
    ## Cortesía de chatGPT (desde linea siguiente hasta el final de esta celda):
    ## save images for later display
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)

    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)  

writer.close() 
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

Let's zoom into the loss function at different places

In [ ]:
np.random.seed(cg.seed)

w_grid_2d = np.linspace(-6, 6, 120)
b_grid_2d = np.linspace(-6, 6, 120)
W, B = np.meshgrid(w_grid_2d, b_grid_2d)

BCE_surf = np.zeros_like(W)
Brier_surf = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        w_ij = np.array([[W[i, j]]])
        b_ij = np.array([[B[i, j]]])
        y_ij = computation_graph_sigmoid(x_data, w_ij, b_ij)
        BCE_surf[i, j] = np.sum(bce_loss_function(t_data, y_ij, clip=1e12))
        Brier_surf[i, j] = np.sum(brier_loss_function(t_data, y_ij))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, surf, title in zip(axes, [BCE_surf, Brier_surf], ["BCE", "Brier"]):
    cf = ax.contourf(W, B, surf, levels=30, cmap="viridis")
    ax.set_xlabel("$w$")
    ax.set_ylabel("$b$")
    ax.set_title(f"{title} loss surface")
    fig.colorbar(cf, ax=ax)

# ground-truth optima found via gradient descent later in this notebook
axes[0].plot(1.1246184, 0.12033531, 'r*', markersize=14, label='global optimum')
axes[1].plot(0.85201174, 0.2264343, 'r*', markersize=14, label='local optimum')
axes[0].legend()
axes[1].legend()
plt.tight_layout()


### Model optimization

As always, we can find this minimum by taking the derivative and setting its value to zero. To do so, we can express, as always, both losses as a composition of functions, exactly as we did for the squared and absolute losses in the [regression theory](1_Regression_Shallow_theory.ipynb). With $\wvec=[w,b]\in\mathbb{R}^2$ and $\Xmat\in\mathbb{R}^{N\times2}$, the first two stages are shared by both losses:

$$
\begin{align*}
\zvec &= \Xmat\wvec && \mathbb{R}^2 \to \mathbb{R}^N\\
\yvec &= \frac{1}{1+e^{-\zvec}} && \mathbb{R}^N \to \mathbb{R}^N \quad\quad \text{element-wise}
\end{align*}
$$

with Jacobians

$$
\begin{split}
J_\zvec &= \Xmat \in \mathbb{R}^{N\times2}\\
J_\yvec &= \diag\pare{\yvec\circ(\onevec-\yvec)} \in \mathbb{R}^{N\times N}
\end{split}
$$

**BCE.** The remaining two stages are:

$$
\begin{align*}
\cvec &= -\bra{\tvec\circ\log\yvec + (\onevec-\tvec)\circ\log(\onevec-\yvec)} && \mathbb{R}^N \to \mathbb{R}^N \quad\quad \text{element-wise}\\
L_{\text{BCE}} &= \onevect\cvec && \mathbb{R}^N \to \mathbb{R}
\end{align*}
$$

with Jacobians

$$
\begin{split}
J_\cvec &= \diag\pare{(\yvec-\tvec)\circ\yvec^{-1}\circ(\onevec-\yvec)^{-1}} \in \mathbb{R}^{N\times N}\\
J_{L_{\text{BCE}}} &= \onevect \in \mathbb{R}^{1\times N}
\end{split}
$$

($\yvec^{-1}$ and $(\onevec-\yvec)^{-1}$ denoting the vectors with element-wise entries inverted). Multiplying the Jacobians of every stage:

$$
\begin{split}
J_\wvec &= J_{L_{\text{BCE}}}J_\cvec J_\yvec J_\zvec \\
&=\onevect\diag\pare{(\yvec-\tvec)\circ\yvec^{-1}\circ(\onevec-\yvec)^{-1}}\diag\pare{\yvec\circ(\onevec-\yvec)}\Xmat
\end{split}
$$

Now, note that the following identity holds $\diag(\xvec \circ \yvec)=\diag(\xvec) \cdot \diag(\yvec)$. Thus $\yvec^{-1}\circ(\onevec-\yvec)^{-1}$ cancels out with $\yvec\circ(\onevec-\yvec)$. This leaves the following gradient and Jacobians.

$$
\begin{split}
J_\wvec &= \onevect\diag(\yvec-\tvec)\Xmat \in \mathbb{R}^{1\times 2}\\
\nabla_\wvec L_{\text{BCE}} &= \Xmatt\diag(\yvec-\tvec)\onevec \in \mathbb{R}^{2\times 1}\\
                             &= \Xmatt(\yvec-\tvec) 
\end{split}
$$

We will re-derive this same expression next via the direct (scalar) chain rule.

**Brier.** Following the same recipe as for the squared loss in the [regression theory](1_Regression_Shallow_theory.ipynb), the remaining two stages collapse into a single element-wise squared residual:

$$
\begin{align*}
\cvec &= (\tvec-\yvec)^2 && \mathbb{R}^N \to \mathbb{R}^N \quad\quad \text{element-wise}\\
L_{\text{Brier}} &= \onevect\cvec && \mathbb{R}^N \to \mathbb{R}
\end{align*}
$$

with Jacobians

$$
\begin{split}
J_\cvec &= -2\diag(\tvec-\yvec) \in \mathbb{R}^{N\times N}\\
J_{L_{\text{Brier}}} &= \onevect \in \mathbb{R}^{1\times N}
\end{split}
$$

Multiplying every stage together:

$$
\begin{split}
J_\wvec &= J_{L_{\text{Brier}}}J_\cvec J_\yvec J_\zvec\\
&= \onevect\cdot(-2)\diag(\tvec-\yvec)\cdot\diag\pare{\yvec\circ(\onevec-\yvec)}\cdot\Xmat\\
&= -2\onevect\diag\pare{(\tvec-\yvec)\circ\yvec\circ(\onevec-\yvec)}\Xmat \in \mathbb{R}^{1\times2}
\end{split}
$$

so that

$$
\begin{split}
\nabla_\wvec L_{\text{Brier}} &= -2\Xmatt\diag\pare{(\tvec-\yvec)\circ\yvec\circ(\onevec-\yvec)}\onevec \in \mathbb{R}^{2\times1}\\
&= -2\Xmatt\bra{(\tvec-\yvec)\circ\yvec\circ(\onevec-\yvec)}
\end{split}
$$

### Optimization via gradient descent

These gradients do not admit closed form solution when equating them to 0. Thus, the optimal value can only be obtained by running numerical methods. In this case, the most efficient way is gradient descent since there is no closed-form coordinate update on $b$ or $w$ as well.

Let's now actually run gradient descent. As in the [regression assessment](1_Regression_assesment_1.ipynb), we first fix $b=0.5$ and optimize $w$ alone, and then optimize both jointly.

#### BCE

In [ ]:
## ========================== ##
## ==== Gradient Descent ==== ##
## ========================== ##
loss_type = 'bce'

if loss_type not in ['bce', 'brier']:
    raise RuntimeError("Invalid loss type choose from bce or brier")

## ======================== ##
## Simulation configuration ##
## ======================== ##
fixed_bias = cg.fixed_bias

## fix seed so that randomness is controlled.
np.random.seed(cg.seed)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

# domain over where we want to plot the function implemented by the NNet
x_range = np.linspace(cg.data_x_range_l,cg.data_x_range_u, N_points_domain).reshape((N_points_domain,1))

## ================ ##
## For plot display ##
## ================ ##
## create figure box
fig = plt.figure(figsize = (10,7))
gs = fig.add_gridspec(2, 2)
ax11 = fig.add_subplot(gs[0,0])
ax13 = fig.add_subplot(gs[1,0])
ax12 = fig.add_subplot(gs[0,1])
fig.subplots_adjust(wspace=0.5, hspace=0.5)

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

## =============================================
## Specify input output to the computation graph
D_in = 1
D_out = 1

## ============================================
## Get the loss function over which we optimize
w_range = np.linspace(cg.losses_named[loss_type].w_range_l,cg.losses_named[loss_type].w_range_u,cg.N_models_simulation).reshape((cg.N_models_simulation,D_in,D_out))

## get predictions for each model
y_data = computation_graph_sigmoid(x_data, w_range, fixed_bias)

## compute loss
if loss_type == 'bce':
    loss_range = cg.losses_named[loss_type].loss_fun(np.tile(t_data,(cg.N_models_simulation,1,1)), y_data)
else:
    loss_range = cg.losses_named[loss_type].loss_fun(t_data, y_data)

## accumulate loss per datapoint
loss_acc_range = np.sum(loss_range, axis = 1)

## squeeze and display
loss_acc_range = np.squeeze(loss_acc_range)
w_range = np.squeeze(w_range)

## Display different models sequentially, alongside its loss.

# display loss function
ax11.plot(w_range, loss_acc_range, color = 'C0')
ax11.set_xlabel('Weight')
ax11.set_ylabel('Loss')

# Initialize parameters
w = np.array([cg.losses_named[loss_type].w_init]).reshape(D_in,D_out)

## gradient descent parameters
lr = 0.9 # try 0.1, 0.01, 0.15, 0.21 to show: fast convergence, slow convergence, convergence with bumping, divergence
epochs = 10

loss_history = []

for e in range(epochs):

    ## forward plus backward
    grad_w, _ = cg.losses_named[loss_type].grad_loss_fun(x_data,t_data, w, fixed_bias)
    
    ## compute function at current parameter value
    function = computation_graph_sigmoid(x_range, w, fixed_bias)

    ## compute predictions at current parameter value
    y_data = computation_graph_sigmoid(x_data, w, fixed_bias)

    ## compute loss at current parameter value
    loss = cg.losses_named[loss_type].loss_fun(t_data,y_data)    
    loss_acc = np.sum(loss)
    loss_history.append(loss_acc)

    ## get the gradient function at the point w (tangent at the point)
    gradient_function_w_at_current_w = grad_w * w_range + loss_acc - grad_w * w
    
    ## compute loss on updated parameters
    w_n = w-lr*grad_w
    
    ## function on new parameters
    function_n = computation_graph_sigmoid(x_range, w_n, fixed_bias)
    
    ## predictions with new parameters
    y_data_n = computation_graph_sigmoid(x_data, w_n, fixed_bias)

    ## compute loss at current parameter value
    loss_n = cg.losses_named[loss_type].loss_fun(t_data, y_data_n)
    loss_acc_n = np.sum(loss_n)
    
    ## ============= ##
    ## ============= ##
    ## START DRAWING ##
    ## ============= ##
    ## ============= ##
    # Clear previous data
    ax11.clear()
    ax12.clear()
    ax13.clear()

    ## =========================== ##
    ## loss evolution picture     ##
    ax13.plot(range(1, len(loss_history) + 1), loss_history, 'o-', color = 'C0')
    ax13.set_xlim([1, epochs])
    ax13.set_xlabel('Epoch')
    ax13.set_ylabel('Loss')
    ax13.set_title(f'{loss_type} loss evolution')

    
    w_plot = np.squeeze(w)
    w_plot_n = np.squeeze(w_n)
    grad_w_plot = np.squeeze(grad_w)
    x_data_plot = np.squeeze(x_data)
    t_data_plot = np.squeeze(t_data)
    y_data_plot = np.squeeze(y_data)
    y_data_plot_n = np.squeeze(y_data_n)
    loss_plot = np.squeeze(loss)
    loss_plot_n = np.squeeze(loss_n)

    ## =========================== ##
    ## prediction function picture ##
    ax12.plot(x_range,function, color = 'C1', label = 'function: y = w*x')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot,loss_plot)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C1', label = 'network prediction')
        else:
            ax12.plot(xi,yi, 'x', color = 'C1')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C1", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C1" ) 

    # label function with the weight at that moment
    ax12.text(x_range[-20],function[-20], f'w = {w_plot:.2f}', color = 'k', fontsize = 12)
    
    ax12.text(0.5, 1.6, f"Iteration {e}, {loss_type} loss = {loss_acc:.2f}", fontsize=12, va='bottom', color = f"C1" ) 
    ax12.set_xlabel(cg.data_x_name)
    ax12.set_ylabel(cg.data_y_name)
    ax12.set_ylim([cg.data_y_lim_l_gd_pred_fun,cg.data_y_lim_u_gd_pred_fun])
    ax12.legend()
    
    ## ===================== ##
    ## loss function picture ##
    ## 0. label and axis limits
    ax11.set_xlabel('Weight')
    ax11.set_ylabel('Loss')
    ax11.set_ylim([cg.losses_named[loss_type].gd_loss_fun_y_lim_l,cg.losses_named[loss_type].gd_loss_fun_y_lim_u])
    ax11.set_xlim([cg.losses_named[loss_type].gd_loss_fun_x_lim_l,cg.losses_named[loss_type].gd_loss_fun_x_lim_u])
          
    ## 1. display loss function
    ax11.plot(w_range, loss_acc_range, color = 'C0', label = 'loss', zorder = 20)    
    
    ## 2. display current weight
    ax11.plot(w_plot, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, '*', color = 'C1', label = 'current weight', zorder = 50, markersize = 10)
    ax11.text(w_plot + 1, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y , f"w = {w_plot:.2f}", fontsize=12, va='bottom', color = f"C1" , zorder = 50)
    ax11.legend(loc = 'upper left')    
        
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## animation by drawing horizontal lines on current parameter and updated parameter values
    ax11.vlines(np.squeeze(w), ymin=cg.losses_named[loss_type].gd_loss_fun_display_param_y, ymax=loss_acc, color='k', linestyles='dotted', zorder = -50)

    ## 3. display current loss
    ax11.plot(w_plot, loss_acc, 'o', color = 'C0', label = 'loss at current weight', zorder = 20)
    ax11.text(w_plot + 0.5, loss_acc , f"loss = {loss_acc:.2f}", fontsize=12, va='bottom', color = "C0" , zorder = 50)
    ax11.legend(loc = 'upper left')
    
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## 4. display the gradient function
    ax11.plot(w_range, np.squeeze(gradient_function_w_at_current_w), color = 'C2', label = 'gradient function: f(w) = grad_w * w + loss - grad_w * w', zorder = 20)
    ax11.text(w_range[-1], np.squeeze(gradient_function_w_at_current_w)[-1], f"grad_w = {grad_w_plot:.2f}", fontsize=12, va='bottom', color = f"C2" , zorder = 200) 
    ax11.legend(loc = 'upper left')
    
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## draw rest of lines to show update
    ax11.hlines(y = loss_acc, xmin=w_plot_n, xmax=w_plot, color='k', linestyles='dotted', zorder = -50)
    
    writer.append_data(frame)
    
    ax11.vlines(w_plot_n, ymin=cg.losses_named[loss_type].gd_loss_fun_display_param_y , ymax=loss_acc, color='k', linestyles='dotted', zorder = -50)
    
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 

    ## 5. display new weight
    ax11.plot(w_plot_n, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, '*', color = 'C3', label = 'updated weight: w_new = w - lr*grad_w', zorder = 200, markersize = 10)
    ax11.text(w_plot_n, cg.losses_named[loss_type].gd_loss_fun_display_param_y - 10*cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, f"w_new = {w_plot:.2f} -{lr:.2f}*{grad_w_plot:.2f} = {w_plot-lr*grad_w_plot:.2f}", fontsize=12, va='bottom', color = f"C3" , zorder = 200) 
    ax11.legend(loc = 'upper left')
        
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## 6. display updated function
    ax12.plot(x_range,function_n, color = 'C3')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot_n,loss_plot_n)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C3')
        else:
            ax12.plot(xi,yi, 'x', color = 'C3')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C3", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C3" ) 

    # label function with the weight at that moment
    ax12.text(x_range[5],function_n[5], f'w = {w_plot_n:.2f}; b = {fixed_bias:.2f}', color = 'C3', fontsize = 12)
    
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## update parameter with gradient descent, for the next update
    w = w-lr*grad_w
    
writer.close() 
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

#### Brier

In [ ]:
## ========================== ##
## ==== Gradient Descent ==== ##
## ========================== ##
loss_type = 'brier'

if loss_type not in ['bce', 'brier']:
    raise RuntimeError("Invalid loss type choose from bce or brier")

## ======================== ##
## Simulation configuration ##
## ======================== ##
fixed_bias = cg.fixed_bias

## fix seed so that randomness is controlled.
np.random.seed(cg.seed)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

# domain over where we want to plot the function implemented by the NNet
x_range = np.linspace(cg.data_x_range_l,cg.data_x_range_u, N_points_domain).reshape((N_points_domain,1))

## ================ ##
## For plot display ##
## ================ ##
## create figure box
fig = plt.figure(figsize = (10,7))
gs = fig.add_gridspec(2, 2)
ax11 = fig.add_subplot(gs[0,0])
ax13 = fig.add_subplot(gs[1,0])
ax12 = fig.add_subplot(gs[0,1])
fig.subplots_adjust(wspace=0.5, hspace=0.5)

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

## =============================================
## Specify input output to the computation graph
D_in = 1
D_out = 1

## ============================================
## Get the loss function over which we optimize
w_range = np.linspace(cg.losses_named[loss_type].w_range_l,cg.losses_named[loss_type].w_range_u,cg.N_models_simulation).reshape((cg.N_models_simulation,D_in,D_out))

## get predictions for each model
y_data = computation_graph_sigmoid(x_data, w_range, fixed_bias)

## compute loss
if loss_type == 'bce':
    loss_range = cg.losses_named[loss_type].loss_fun(np.tile(t_data,(cg.N_models_simulation,1,1)), y_data)
else:
    loss_range = cg.losses_named[loss_type].loss_fun(t_data, y_data)

## accumulate loss per datapoint
loss_acc_range = np.sum(loss_range, axis = 1)

## squeeze and display
loss_acc_range = np.squeeze(loss_acc_range)
w_range = np.squeeze(w_range)

## Display different models sequentially, alongside its loss.

# display loss function
ax11.plot(w_range, loss_acc_range, color = 'C0')
ax11.set_xlabel('Weight')
ax11.set_ylabel('Loss')

# Initialize parameters
w = np.array([cg.losses_named[loss_type].w_init]).reshape(D_in,D_out)

## gradient descent parameters
lr = 0.9 # try 0.1, 0.01, 0.15, 0.21 to show: fast convergence, slow convergence, convergence with bumping, divergence
epochs = 10

loss_history = []

for e in range(epochs):

    ## forward plus backward
    grad_w, _ = cg.losses_named[loss_type].grad_loss_fun(x_data,t_data, w, fixed_bias)
    
    ## compute function at current parameter value
    function = computation_graph_sigmoid(x_range, w, fixed_bias)

    ## compute predictions at current parameter value
    y_data = computation_graph_sigmoid(x_data, w, fixed_bias)

    ## compute loss at current parameter value
    loss = cg.losses_named[loss_type].loss_fun(t_data,y_data)    
    loss_acc = np.sum(loss)
    loss_history.append(loss_acc)

    ## get the gradient function at the point w (tangent at the point)
    gradient_function_w_at_current_w = grad_w * w_range + loss_acc - grad_w * w
    
    ## compute loss on updated parameters
    w_n = w-lr*grad_w
    
    ## function on new parameters
    function_n = computation_graph_sigmoid(x_range, w_n, fixed_bias)
    
    ## predictions with new parameters
    y_data_n = computation_graph_sigmoid(x_data, w_n, fixed_bias)

    ## compute loss at current parameter value
    loss_n = cg.losses_named[loss_type].loss_fun(t_data, y_data_n)
    loss_acc_n = np.sum(loss_n)
    
    ## ============= ##
    ## ============= ##
    ## START DRAWING ##
    ## ============= ##
    ## ============= ##
    # Clear previous data
    ax11.clear()
    ax12.clear()
    ax13.clear()

    ## =========================== ##
    ## loss evolution picture     ##
    ax13.plot(range(1, len(loss_history) + 1), loss_history, 'o-', color = 'C0')
    ax13.set_xlim([1, epochs])
    ax13.set_xlabel('Epoch')
    ax13.set_ylabel('Loss')
    ax13.set_title(f'{loss_type} loss evolution')

    
    w_plot = np.squeeze(w)
    w_plot_n = np.squeeze(w_n)
    grad_w_plot = np.squeeze(grad_w)
    x_data_plot = np.squeeze(x_data)
    t_data_plot = np.squeeze(t_data)
    y_data_plot = np.squeeze(y_data)
    y_data_plot_n = np.squeeze(y_data_n)
    loss_plot = np.squeeze(loss)
    loss_plot_n = np.squeeze(loss_n)

    ## =========================== ##
    ## prediction function picture ##
    ax12.plot(x_range,function, color = 'C1', label = 'function: y = w*x')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot,loss_plot)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C1', label = 'network prediction')
        else:
            ax12.plot(xi,yi, 'x', color = 'C1')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C1", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C1" ) 

    # label function with the weight at that moment
    ax12.text(x_range[-20],function[-20], f'w = {w_plot:.2f}', color = 'k', fontsize = 12)
    
    ax12.text(0.5, 1.6, f"Iteration {e}, {loss_type} loss = {loss_acc:.2f}", fontsize=12, va='bottom', color = f"C1" ) 
    ax12.set_xlabel(cg.data_x_name)
    ax12.set_ylabel(cg.data_y_name)
    ax12.set_ylim([cg.data_y_lim_l_gd_pred_fun,cg.data_y_lim_u_gd_pred_fun])
    ax12.legend()
    
    ## ===================== ##
    ## loss function picture ##
    ## 0. label and axis limits
    ax11.set_xlabel('Weight')
    ax11.set_ylabel('Loss')
    ax11.set_ylim([cg.losses_named[loss_type].gd_loss_fun_y_lim_l,cg.losses_named[loss_type].gd_loss_fun_y_lim_u])
    ax11.set_xlim([cg.losses_named[loss_type].gd_loss_fun_x_lim_l,cg.losses_named[loss_type].gd_loss_fun_x_lim_u])
          
    ## 1. display loss function
    ax11.plot(w_range, loss_acc_range, color = 'C0', label = 'loss', zorder = 20)    
    
    ## 2. display current weight
    ax11.plot(w_plot, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, '*', color = 'C1', label = 'current weight', zorder = 50, markersize = 10)
    ax11.text(w_plot + 1, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y , f"w = {w_plot:.2f}", fontsize=12, va='bottom', color = f"C1" , zorder = 50)
    ax11.legend(loc = 'upper left')    
        
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## animation by drawing horizontal lines on current parameter and updated parameter values
    ax11.vlines(np.squeeze(w), ymin=cg.losses_named[loss_type].gd_loss_fun_display_param_y, ymax=loss_acc, color='k', linestyles='dotted', zorder = -50)

    ## 3. display current loss
    ax11.plot(w_plot, loss_acc, 'o', color = 'C0', label = 'loss at current weight', zorder = 20)
    ax11.text(w_plot + 0.5, loss_acc , f"loss = {loss_acc:.2f}", fontsize=12, va='bottom', color = "C0" , zorder = 50)
    ax11.legend(loc = 'upper left')
    
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## 4. display the gradient function
    ax11.plot(w_range, np.squeeze(gradient_function_w_at_current_w), color = 'C2', label = 'gradient function: f(w) = grad_w * w + loss - grad_w * w', zorder = 20)
    ax11.text(w_range[-1], np.squeeze(gradient_function_w_at_current_w)[-1], f"grad_w = {grad_w_plot:.2f}", fontsize=12, va='bottom', color = f"C2" , zorder = 200) 
    ax11.legend(loc = 'upper left')
    
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## draw rest of lines to show update
    ax11.hlines(y = loss_acc, xmin=w_plot_n, xmax=w_plot, color='k', linestyles='dotted', zorder = -50)
    
    writer.append_data(frame)
    
    ax11.vlines(w_plot_n, ymin=cg.losses_named[loss_type].gd_loss_fun_display_param_y , ymax=loss_acc, color='k', linestyles='dotted', zorder = -50)
    
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 

    ## 5. display new weight
    ax11.plot(w_plot_n, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, '*', color = 'C3', label = 'updated weight: w_new = w - lr*grad_w', zorder = 200, markersize = 10)
    ax11.text(w_plot_n, cg.losses_named[loss_type].gd_loss_fun_display_param_y - 10*cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, f"w_new = {w_plot:.2f} -{lr:.2f}*{grad_w_plot:.2f} = {w_plot-lr*grad_w_plot:.2f}", fontsize=12, va='bottom', color = f"C3" , zorder = 200) 
    ax11.legend(loc = 'upper left')
        
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## 6. display updated function
    ax12.plot(x_range,function_n, color = 'C3')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot_n,loss_plot_n)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C3')
        else:
            ax12.plot(xi,yi, 'x', color = 'C3')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C3", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C3" ) 

    # label function with the weight at that moment
    ax12.text(x_range[5],function_n[5], f'w = {w_plot_n:.2f}; b = {fixed_bias:.2f}', color = 'C3', fontsize = 12)
    
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## update parameter with gradient descent, for the next update
    w = w-lr*grad_w
    
writer.close() 
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

Let's now run gradient descent on the full model for both losses.

#### BCE

In [ ]:
## ======================================================================================= ##
## display loss as a function of weight and bias parameter (loss incurred by each network) ##
## ======================================================================================= ##
## Let's see the associated loss to each possible function but seeing the loss
## as a function of the weight and bias parameter. 
## We show two different losses: squared (top) and absolute ( bottom )
loss_type = 'bce'

if loss_type not in ['brier', 'bce']:
    raise RuntimeError("Invalid loss type choose from bce or brier")

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")       
    
## create figure box
fig = plt.figure(figsize = (10,7))
gs = fig.add_gridspec(2, 2)
ax11 = fig.add_subplot(gs[0,0], projection='3d')
ax11.view_init(elev=30, azim=cg.losses_named[loss_type].full_loss_display_azim)
ax12 = fig.add_subplot(gs[0,1])
ax13 = fig.add_subplot(gs[1,0])
fig.subplots_adjust(wspace=0.5, hspace=0.5)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

# domain over where we want to plot the function implemented by the NNet
x_range = np.linspace(cg.data_x_range_l,cg.data_x_range_u, N_points_domain).reshape((N_points_domain,1))


## ===========================================
## Neural network specification for each layer

# neurons of input layer
n_in = 1
# neurons of output layer
n_out = 1

## ================================================================================
## Create several possible functions that our specific neural network can implement
## first of all draw loss function against a set of parameters

## To do so we need a mesh
w_mesh, b_mesh = np.meshgrid(
    np.linspace(cg.losses_named[loss_type].w_range_l_2d,cg.losses_named[loss_type].w_range_u_2d,cg.N_models_simulation),
    np.linspace(cg.losses_named[loss_type].b_range_l_2d,cg.losses_named[loss_type].b_range_u_2d,cg.N_models_simulation)
)

# reshape x_data and t_data for computations. t_data uses broadcasting.
x_data_expanded = x_data[:,np.newaxis]
t_data_expanded = t_data[:,np.newaxis]

# compute linear projection at all pairs of points
y_data_expanded = activation_function_sigmoid(w_mesh*x_data_expanded + b_mesh)

# compute loss
if loss_type == 'bce':
    loss_acc_mesh = np.sum(cg.losses_named[loss_type].loss_fun(np.squeeze(t_data_expanded), y_data_expanded), axis = 0)
else:
    loss_acc_mesh = np.sum(cg.losses_named[loss_type].loss_fun(t_data_expanded, y_data_expanded), axis = 0)

max_val = np.max(loss_acc_mesh[loss_acc_mesh != np.inf ])
min_val = np.min(loss_acc_mesh[loss_acc_mesh != -np.inf ])
loss_acc_mesh[loss_acc_mesh == np.inf ] = max_val
loss_acc_mesh[loss_acc_mesh == -np.inf ] = min_val
   
ax11.plot_surface(w_mesh, b_mesh, loss_acc_mesh, cmap = 'gray')
ax11.set_xlabel('weight values')
ax11.set_ylabel('bias values')
ax11.set_zlabel(f'{loss_type} loss function')

# Initialize parameters
w = np.array([cg.losses_named[loss_type].w_init_full]).reshape(n_in,n_out)
b = np.array([cg.losses_named[loss_type].b_init_full])

"""
**optimal parameters for bce loss (global)**
weight = 1.1246184
bias = 0.12033531


**optimal parameters for mse loss (local)**
weight = 0.85201174
bias = 0.2264343
"""

## gradient descent parameters
lr = 3
epochs = 20

loss_history = []

for e in range(epochs):
    
    ## forward plus backward
    grad_w, grad_b = cg.losses_named[loss_type].grad_loss_fun(x_data,t_data, w, b)

    ## compute function at current parameter value
    function = computation_graph_sigmoid(x_range, w, b)

    ## compute predictions at current parameter value
    y_data = computation_graph_sigmoid(x_data, w, b)

    ## compute loss at current parameter value
    loss = cg.losses_named[loss_type].loss_fun(t_data, y_data)
    
    loss_acc = np.sum(loss)
    loss_history.append(loss_acc)
    
    ## compute loss on updated parameters
    w_n = w-lr*grad_w
    b_n = b-lr*grad_b
    
    ## function on new parameters
    function_n = computation_graph_sigmoid(x_range, w_n, b_n)
    
    ## predictions with new parameters
    y_data_n = computation_graph_sigmoid(x_data, w_n, b_n)

    ## compute loss at current parameter value
    loss_n = cg.losses_named[loss_type].loss_fun(t_data, y_data_n)

    loss_acc_n = np.sum(loss_n)
    
    ## ============= ##
    ## ============= ##
    ## START DRAWING ##
    ## ============= ##
    ## ============= ##
    # Clear previous data
    ax11.clear()
    ax12.clear()
    ax13.clear()

    ## =========================== ##
    ## loss evolution picture     ##
    ax13.plot(range(1, len(loss_history) + 1), loss_history, 'o-', color = 'C0')
    ax13.set_xlim([1, epochs])
    ax13.set_xlabel('Epoch')
    ax13.set_ylabel('Loss')
    ax13.set_title(f'{loss_type} loss evolution')
    
    w_plot = np.squeeze(w)
    b_plot = np.squeeze(b)
    
    w_plot_n = np.squeeze(w_n)
    b_plot_n = np.squeeze(b_n)

    x_data_plot = np.squeeze(x_data)
    t_data_plot = np.squeeze(t_data)
    y_data_plot = np.squeeze(y_data)
    y_data_plot_n = np.squeeze(y_data_n)
    loss_plot = np.squeeze(loss)
    loss_plot_n = np.squeeze(loss_n)
    
    ## ================ ##
    ## function picture ##
    ax12.plot(x_range,function, color = 'C1', label = 'function: y = w*x + b')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot,loss_plot)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C1', label = 'network prediction')
        else:
            ax12.plot(xi,yi, 'x', color = 'C1')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C1", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C1" ) 

    # label function with the weight at that moment
    ax12.text(x_range[-20],function[-20], f'w = {w_plot:.2f}; b = {b_plot:.2f}', color = 'C1', fontsize = 12)
    
    ax12.text(0, 1.2, f"Iteration {e}, {cg.losses_named[loss_type].loss_name} = {loss_acc:.2f}", fontsize=12, va='bottom', color = f"C1" ) 
    ax12.set_xlabel(cg.data_x_name)
    ax12.set_ylabel(cg.data_y_name)
    ax12.set_ylim([cg.data_y_lim_l_gd_pred_fun,cg.data_y_lim_u_gd_pred_fun])
    ax12.legend()
    
    ## ===================== ##
    ## loss function picture ##
    
    ## 1. display loss function
    ax11.plot_surface(w_mesh, b_mesh, loss_acc_mesh, cmap = 'gray')
    ax11.set_xlabel('weight values')
    ax11.set_ylabel('bias values')
    ax11.set_zlabel(f'{cg.losses_named[loss_type].loss_name} loss function')
    
    ## 2. display current weight
    ax11.plot(w_plot, b_plot, loss_acc, 'o', color = 'C1', label = 'current weight', zorder = 50, markersize = 5)
    ax11.text(w_plot, b_plot, loss_acc, f"(w,b) = ({w_plot:.2f},{b_plot:.2f})", fontsize=12, va='bottom', color = f"C1" , zorder = 50)
    ax11.legend()
    
    ## save image frame
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)
    
    ## 3. display the gradient arrow and updated weight
    a = Arrow3D([w_plot, w_plot_n], [b_plot, b_plot_n], [loss_acc, loss_acc_n],
                mutation_scale=20, lw=0.5, arrowstyle="-|>", color="C0")
    ax11.add_artist(a)
    ax11.legend()

    ## 4. display new weight
    ax11.plot(w_plot_n, b_plot_n, loss_acc_n, 'o', color = 'C3', label = 'updated weight', zorder = 50, markersize = 5)
    ax11.text(w_plot_n, b_plot_n, loss_acc_n, f"(w_new,b_new) = ({w_plot_n:.2f},{b_plot_n:.2f})", fontsize=12, va='bottom', color = f"C3" , zorder = 50)
    ax11.legend()
    
    ## save image frame
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)
    
    ## 6. display updated function
    ax12.plot(x_range,function_n, color = 'C3')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot_n,loss_plot_n)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C3')
        else:
            ax12.plot(xi,yi, 'x', color = 'C3')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C3", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C3" ) 

    # label function with the weight at that moment
    ax12.text(x_range[5],function_n[5], f'w = {w_plot_n:.2f}; b = {b_plot_n:.2f}', color = 'C3', fontsize = 12)
    
    ## save image frame
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## update parameter with gradient descent, for the next update
    w = w-lr*grad_w
    b = b-lr*grad_b
    
writer.close() 
plt.close()


In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

#### Brier

In [ ]:
## ======================================================================================= ##
## display loss as a function of weight and bias parameter (loss incurred by each network) ##
## ======================================================================================= ##
## Let's see the associated loss to each possible function but seeing the loss
## as a function of the weight and bias parameter. 
## We show two different losses: squared (top) and absolute ( bottom )
loss_type = 'brier'

if loss_type not in ['brier', 'bce']:
    raise RuntimeError("Invalid loss type choose from bce or brier")

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")       
    
## create figure box
fig = plt.figure(figsize = (10,7))
gs = fig.add_gridspec(2, 2)
ax11 = fig.add_subplot(gs[0,0], projection='3d')
ax11.view_init(elev=30, azim=cg.losses_named[loss_type].full_loss_display_azim)
ax12 = fig.add_subplot(gs[0,1])
ax13 = fig.add_subplot(gs[1,0])
fig.subplots_adjust(wspace=0.5, hspace=0.5)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

# domain over where we want to plot the function implemented by the NNet
x_range = np.linspace(cg.data_x_range_l,cg.data_x_range_u, N_points_domain).reshape((N_points_domain,1))


## ===========================================
## Neural network specification for each layer

# neurons of input layer
n_in = 1
# neurons of output layer
n_out = 1

## ================================================================================
## Create several possible functions that our specific neural network can implement
## first of all draw loss function against a set of parameters

## To do so we need a mesh
w_mesh, b_mesh = np.meshgrid(
    np.linspace(cg.losses_named[loss_type].w_range_l_2d,cg.losses_named[loss_type].w_range_u_2d,cg.N_models_simulation),
    np.linspace(cg.losses_named[loss_type].b_range_l_2d,cg.losses_named[loss_type].b_range_u_2d,cg.N_models_simulation)
)

# reshape x_data and t_data for computations. t_data uses broadcasting.
x_data_expanded = x_data[:,np.newaxis]
t_data_expanded = t_data[:,np.newaxis]

# compute linear projection at all pairs of points
y_data_expanded = activation_function_sigmoid(w_mesh*x_data_expanded + b_mesh)

# compute loss
if loss_type == 'bce':
    loss_acc_mesh = np.sum(cg.losses_named[loss_type].loss_fun(np.squeeze(t_data_expanded), y_data_expanded), axis = 0)
else:
    loss_acc_mesh = np.sum(cg.losses_named[loss_type].loss_fun(t_data_expanded, y_data_expanded), axis = 0)

max_val = np.max(loss_acc_mesh[loss_acc_mesh != np.inf ])
min_val = np.min(loss_acc_mesh[loss_acc_mesh != -np.inf ])
loss_acc_mesh[loss_acc_mesh == np.inf ] = max_val
loss_acc_mesh[loss_acc_mesh == -np.inf ] = min_val
   
ax11.plot_surface(w_mesh, b_mesh, loss_acc_mesh, cmap = 'gray')
ax11.set_xlabel('weight values')
ax11.set_ylabel('bias values')
ax11.set_zlabel(f'{loss_type} loss function')

# Initialize parameters
w = np.array([cg.losses_named[loss_type].w_init_full]).reshape(n_in,n_out)
b = np.array([cg.losses_named[loss_type].b_init_full])

"""
**optimal parameters for bce loss (global)**
weight = 1.1246184
bias = 0.12033531


**optimal parameters for mse loss (local)**
weight = 0.85201174
bias = 0.2264343
"""

## gradient descent parameters
lr = 3
epochs = 20

loss_history = []

for e in range(epochs):
    
    ## forward plus backward
    grad_w, grad_b = cg.losses_named[loss_type].grad_loss_fun(x_data,t_data, w, b)

    ## compute function at current parameter value
    function = computation_graph_sigmoid(x_range, w, b)

    ## compute predictions at current parameter value
    y_data = computation_graph_sigmoid(x_data, w, b)

    ## compute loss at current parameter value
    loss = cg.losses_named[loss_type].loss_fun(t_data, y_data)
    
    loss_acc = np.sum(loss)
    loss_history.append(loss_acc)
    
    ## compute loss on updated parameters
    w_n = w-lr*grad_w
    b_n = b-lr*grad_b
    
    ## function on new parameters
    function_n = computation_graph_sigmoid(x_range, w_n, b_n)
    
    ## predictions with new parameters
    y_data_n = computation_graph_sigmoid(x_data, w_n, b_n)

    ## compute loss at current parameter value
    loss_n = cg.losses_named[loss_type].loss_fun(t_data, y_data_n)

    loss_acc_n = np.sum(loss_n)
    
    ## ============= ##
    ## ============= ##
    ## START DRAWING ##
    ## ============= ##
    ## ============= ##
    # Clear previous data
    ax11.clear()
    ax12.clear()
    ax13.clear()

    ## =========================== ##
    ## loss evolution picture     ##
    ax13.plot(range(1, len(loss_history) + 1), loss_history, 'o-', color = 'C0')
    ax13.set_xlim([1, epochs])
    ax13.set_xlabel('Epoch')
    ax13.set_ylabel('Loss')
    ax13.set_title(f'{loss_type} loss evolution')
    
    w_plot = np.squeeze(w)
    b_plot = np.squeeze(b)
    
    w_plot_n = np.squeeze(w_n)
    b_plot_n = np.squeeze(b_n)

    x_data_plot = np.squeeze(x_data)
    t_data_plot = np.squeeze(t_data)
    y_data_plot = np.squeeze(y_data)
    y_data_plot_n = np.squeeze(y_data_n)
    loss_plot = np.squeeze(loss)
    loss_plot_n = np.squeeze(loss_n)
    
    ## ================ ##
    ## function picture ##
    ax12.plot(x_range,function, color = 'C1', label = 'function: y = w*x + b')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot,loss_plot)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C1', label = 'network prediction')
        else:
            ax12.plot(xi,yi, 'x', color = 'C1')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C1", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C1" ) 

    # label function with the weight at that moment
    ax12.text(x_range[-20],function[-20], f'w = {w_plot:.2f}; b = {b_plot:.2f}', color = 'C1', fontsize = 12)
    
    ax12.text(0, 1.2, f"Iteration {e}, {cg.losses_named[loss_type].loss_name} = {loss_acc:.2f}", fontsize=12, va='bottom', color = f"C1" ) 
    ax12.set_xlabel(cg.data_x_name)
    ax12.set_ylabel(cg.data_y_name)
    ax12.set_ylim([cg.data_y_lim_l_gd_pred_fun,cg.data_y_lim_u_gd_pred_fun])
    ax12.legend()
    
    ## ===================== ##
    ## loss function picture ##
    
    ## 1. display loss function
    ax11.plot_surface(w_mesh, b_mesh, loss_acc_mesh, cmap = 'gray')
    ax11.set_xlabel('weight values')
    ax11.set_ylabel('bias values')
    ax11.set_zlabel(f'{cg.losses_named[loss_type].loss_name} loss function')
    
    ## 2. display current weight
    ax11.plot(w_plot, b_plot, loss_acc, 'o', color = 'C1', label = 'current weight', zorder = 50, markersize = 5)
    ax11.text(w_plot, b_plot, loss_acc, f"(w,b) = ({w_plot:.2f},{b_plot:.2f})", fontsize=12, va='bottom', color = f"C1" , zorder = 50)
    ax11.legend()
    
    ## save image frame
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)
    
    ## 3. display the gradient arrow and updated weight
    a = Arrow3D([w_plot, w_plot_n], [b_plot, b_plot_n], [loss_acc, loss_acc_n],
                mutation_scale=20, lw=0.5, arrowstyle="-|>", color="C0")
    ax11.add_artist(a)
    ax11.legend()

    ## 4. display new weight
    ax11.plot(w_plot_n, b_plot_n, loss_acc_n, 'o', color = 'C3', label = 'updated weight', zorder = 50, markersize = 5)
    ax11.text(w_plot_n, b_plot_n, loss_acc_n, f"(w_new,b_new) = ({w_plot_n:.2f},{b_plot_n:.2f})", fontsize=12, va='bottom', color = f"C3" , zorder = 50)
    ax11.legend()
    
    ## save image frame
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)
    
    ## 6. display updated function
    ax12.plot(x_range,function_n, color = 'C3')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot_n,loss_plot_n)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C3')
        else:
            ax12.plot(xi,yi, 'x', color = 'C3')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C3", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C3" ) 

    # label function with the weight at that moment
    ax12.text(x_range[5],function_n[5], f'w = {w_plot_n:.2f}; b = {b_plot_n:.2f}', color = 'C3', fontsize = 12)
    
    ## save image frame
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame) 
    
    ## update parameter with gradient descent, for the next update
    w = w-lr*grad_w
    b = b-lr*grad_b
    
writer.close() 
plt.close()


In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

### Decision Threshold

One interesting thing about these kinds of models is that we might want to make decisions based on the probability assigned by the model. This leads to what are called decision thresholds, which are the set of points where our decision changes from one class to another. As you might imagine, a linear model (even if the whole model is not linear due to the sigmoid) results in a linear boundary. When inputs are a single dimension, this linear boundary is just a point on the $x$ axis. In the next section, where we deal with multiple inputs, we will observe the lines spanned by planes, and the linearity of the decision threshold will be clearer.

To check this, note that we might decide to change from one class to another when the probability assigned by the model is, for instance, $0.5$. Thus we need to solve for the points $x$ where this equality holds. Assume we have optimized our model and have some $\wvec_\text{opt}$ in this case obtained by gradient descent.

$$
\begin{split}
y = \frac{1}{1+e^{-\xvect\wvec}}
\end{split}
$$


Since the sigmoid is monotonically increasing and invertible on its range $(0,1)$, we can solve directly for the pre-sigmoid value $z=\xvect\wvec$ that gives the desired probability, instead of solving for $y$ directly. In particular, the sigmoid takes the value $0.5$ exactly when its input is $0$:

$$
\begin{split}
z=0 \Longleftrightarrow 0.5 = \frac{1}{1+e^{-z}}
\end{split}
$$

so the decision threshold is the set of points $\xvec$ satisfying

$$
\begin{split}
0 = \xvect\wvec_\text{opt}
\end{split}
$$

Note that this equation cannot be solved by $\xvec=0$, since $\xvec=[x,1]$ by construction, ie, it has one of its coordinates set to $1$. If we expand the dot product, the equation is solved at:

$$
\begin{split}
0 &= w\cdot x+b\\
x &= \frac{-b}{w}
\end{split}
$$

Overall, this gives the rule "decide class 1" whenever $\xvect\wvec_\text{opt}\geq 0$. We can obviously want to put a more restrictive threshold for deciding class 1. We just need to check what the corresponding pre-sigmoid value $ a$ is. Then, the boundary will be at $\frac{a-b}{w}$, and the rule will be "decide class 1" whenever $\xvect\wvec_\text{opt}\geq a$

Let's visualize this decision threshold for different probabilities: $p=0.5$ and $p=0.8$. Note that even if we plot a horizontal line, the decision threshold is a point in $y ya cux$.

In [ ]:
# global optimum for the BCE loss on this dataset (found via gradient descent later in this notebook)
w_opt, b_opt = 1.1246184, 0.12033531

y_range_opt = computation_graph_sigmoid(x_range, np.array([[w_opt]]), np.array([[b_opt]]))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, p in zip(axes, [0.5, 0.8]):
    a = np.log(p / (1 - p))  # pre-sigmoid value a such that sigmoid(a) = p
    x_threshold = (a - b_opt) / w_opt

    ax.axvspan(cg.data_x_lim_l, x_threshold, color=color_c0, alpha=0.15, label='decide class 0 (cat)')
    ax.axvspan(x_threshold, cg.data_x_lim_u, color=color_c1, alpha=0.15, label='decide class 1 (dog)')

    ax.plot(x_data[idx_class0], t_data[idx_class0], 'o', color=color_c0, markersize=8, label='class 0 (cat)')
    ax.plot(x_data[idx_class1], t_data[idx_class1], '*', color=color_c1, markersize=10, label='class 1 (dog)')
    ax.plot(x_range, y_range_opt, color='C3', label=f'optimal sigmoid (w={w_opt:.3f}, b={b_opt:.3f})', linewidth = 4)
    ax.axhline(p, color='k', linestyle='--', alpha=0.5)
    ax.axvline(x_threshold, color='k', linestyle='-', linewidth=3, label=f'decision threshold ($p={p}$): x = {x_threshold:.3f}')
    ax.set_xlabel(cg.data_x_name)
    ax.set_ylabel(cg.data_y_name)
    ax.set_xlim([cg.data_x_lim_l, cg.data_x_lim_u])
    ax.set_ylim([cg.data_y_lim_l, cg.data_y_lim_u])
    ax.set_title(f'$p={p}$')
    ax.legend(loc='upper left')
plt.tight_layout()

### Non-linear models

In a similar way to what we have been doing so far, we can create non-linear models using linear basis function models. Note that the data we have been working with is, by construction,  non-linearly separable. Let's see if we can learn to separate the data using a linear model of polinomial basis of high order.

In [ ]:
poly_degree = 5
x_range = np.linspace(cg.data_x_range_l, cg.data_x_range_u, 500).reshape(-1, 1)

X_feat = generate_features(x_data, poly_degree, add_bias=False)        # [x, x^2, ..., x^poly_degree]
x_range_feat = generate_features(x_range, poly_degree, add_bias=False)

np.random.seed(cg.seed)
w = np.random.normal(0, 0.1, size=(poly_degree, 1))
b = np.array([[0.0]])

lr = 0.1
epochs = 3000

loss_history = []
for e in range(epochs):
    y = computation_graph_sigmoid(X_feat, w, b)
    loss_history.append(np.sum(bce_loss_function(t_data, y, clip=1e12)))

    grad_w, grad_b = grad_bce_loss_wrt_sigmoid_model(X_feat, t_data, w, b)
    w = w - lr * grad_w
    b = b - lr * grad_b

print(f"poly_degree={poly_degree}: final BCE loss = {loss_history[-1]:.4f}")

function = computation_graph_sigmoid(x_range_feat, w, b)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(x_data[idx_class0], t_data[idx_class0], 'o', color=color_c0, markersize=8, label='class 0 (cat)')
ax1.plot(x_data[idx_class1], t_data[idx_class1], '*', color=color_c1, markersize=10, label='class 1 (dog)')
ax1.plot(x_range, function, color='C1', label=f'degree-{poly_degree} polynomial model')
ax1.set_xlabel(cg.data_x_name)
ax1.set_ylabel(cg.data_y_name)
ax1.set_xlim([cg.data_x_lim_l, cg.data_x_lim_u])
ax1.set_ylim([cg.data_y_lim_l, cg.data_y_lim_u])
ax1.legend(loc='upper left')

ax2.plot(loss_history, color='C0')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('BCE loss evolution')
plt.tight_layout()

#### Computing the decision boundary for the polynomial model

The rule is unchanged: predict class $1$ whenever $z=\xvect\wvec_\text{opt}\geq0$, only now $\xvec=[\phi_1(x),\dots,\phi_5(x)]=[x,x^2,x^3,x^4,x^5]$ is the vector of polynomial features (with $b$ playing the role of $\phi_0(x)=1$). Since $z$ is now a degree-$5$ polynomial in $x$,

$$
z(x)=w_1x+w_2x^2+w_3x^3+w_4x^4+w_5x^5+b,
$$

the decision threshold is no longer the single point we found in the linear case: it is the set of roots of $z(x)=0$, and a degree-$5$ polynomial can have up to $5$ real roots. Every time $x$ crosses one of these roots, the sign of $z(x)$ flips, so the predicted class can alternate several times as we move along the $x$-axis, instead of switching only once.

There is also a more fundamental difference: unlike the linear case, there is in general no closed-form expression for these roots. By the Abel-Ruffini theorem, polynomials of degree $5$ or higher cannot always be solved by radicals, so the decision boundary has to be found numerically (e.g. with `numpy.roots`) rather than through an explicit formula like the $x=-b/w$ we derived above. This will be the general case for different basis functions, which might result in non-polynomial equations.

Note that to make decisions we do not really need the decision threshold but just the decision rule. If you want to plot it, another option is to just define a grid of points, evaluate the dot product on all of them an check which points satisfy the decision rule.

In [ ]:
# decision rule: predict class 1 wherever y(x) = sigmoid(w^T phi(x) + b) >= 0.5,
# evaluated directly on the grid -- no closed form needed, as noted above.
decision_1 = np.squeeze(function) >= 0.5
x_flat = np.squeeze(x_range)

# the actual boundary points, found numerically (Abel-Ruffini: no closed form for degree >= 5)
poly_coeffs = np.concatenate([w[::-1].flatten(), b.flatten()])   # [w5,...,w1,b], highest degree first
roots = np.roots(poly_coeffs)
real_roots = np.real(roots[np.isreal(roots)])
boundary_points = real_roots[(real_roots >= cg.data_x_lim_l) & (real_roots <= cg.data_x_lim_u)]
print(f"decision boundary points: {sorted(boundary_points)}")

fig, ax = plt.subplots(1, 1, figsize=(9, 5))

ax.fill_between(x_flat, cg.data_y_lim_l, cg.data_y_lim_u, where=decision_1, color=color_c1, alpha=0.15, label='decide class 1 (dog)')
ax.fill_between(x_flat, cg.data_y_lim_l, cg.data_y_lim_u, where=~decision_1, color=color_c0, alpha=0.15, label='decide class 0 (cat)')

for xb in boundary_points:
    ax.axvline(xb, color='k', linewidth=3)

ax.plot(x_data[idx_class0], t_data[idx_class0], 'o', color=color_c0, markersize=8, label='class 0 (cat)')
ax.plot(x_data[idx_class1], t_data[idx_class1], '*', color=color_c1, markersize=10, label='class 1 (dog)')
ax.plot(x_range, function, color='C3', label=f'degree-{poly_degree} polynomial model', linewidth = 4)
ax.axhline(0.5, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel(cg.data_x_name)
ax.set_ylabel(cg.data_y_name)
ax.set_xlim([cg.data_x_lim_l, cg.data_x_lim_u])
ax.set_ylim([cg.data_y_lim_l, cg.data_y_lim_u])
ax.legend(loc='upper left')

## Multivariate binary classification: $f:\mathbb{R}^D\rightarrow\{0,1\}$

Let's now move to $D>1$ inputs. Our new toy dataset has weight ($x_1$) and height ($x_2$) as features, and the same cat/dog label:

$$
\begin{split}
(x^1_1,x^1_2,t^1) &= (0,1,0)\\
(x^2_1,x^2_2,t^2) &= (1.5,2.0,0)\\
(x^3_1,x^3_2,t^3) &= (2,1,0)\\
(x^4_1,x^4_2,t^4) &= (2.5,2,0)\\
(x^5_1,x^5_2,t^5) &= (3,4,1)\\
(x^6_1,x^6_2,t^6) &= (4,5,1)\\
(x^7_1,x^7_2,t^7) &= (5,1,1)\\
\end{split}
$$


The good point of classification problems is that we can still use 2-dimensional plots for illustrative purposes. To be honest, while regression seems easier to understand in one-dimensional settings, classification is easier to see when two features are used. Let's plot our new data.

In [ ]:
cg = cg_rl_R201

x_data = np.array([[0,1],[1.5,2.0],[2,1],[2.5,2],[3,4],[4,5],[5,1]])
t_data = np.array([0,0,0,0,1,1,1]).reshape(-1, 1)

idx_class0 = (t_data == 0).ravel()
idx_class1 = (t_data == 1).ravel()

fig, ax = plt.subplots(1, 1, figsize=(7, 6))
ax.plot(x_data[idx_class0, 0], x_data[idx_class0, 1], 'o', color=cg.color_class0, markersize=8, label=f'class 0 ({cg.name_class0})')
ax.plot(x_data[idx_class1, 0], x_data[idx_class1, 1], '*', color=cg.color_class1, markersize=10, label=f'class 1 ({cg.name_class1})')
ax.set_xlabel(cg.data_x1_name)
ax.set_ylabel(cg.data_x2_name)
ax.set_xlim([cg.data_x1_lim_l, cg.data_x1_lim_u])
ax.set_ylim([cg.data_x2_lim_l, cg.data_x2_lim_u])
ax.legend()


The model can, as always, be written down as:

$$
\begin{split}
z = \xvect\wvec\\
y = \frac{1}{1+e^{-z}}
\end{split}
$$

with the only difference that $\xvect=[x_1,x_2,1]$. We add the new feature. The gradient descent algorithm, the loss, and everything is exactly the same as we have seen up to this point.

Let's visualize some models explaining this data. However, rather than plotting the 3-dimensional sigmoids (or something like this), we directly use 2-dimensional plots showing the decision thresholds alongside the probability assigned by the model. Changes in color intensity indicate more assigned probability in that region. We will talk later about decision thresholds in this setting.

In [ ]:
# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

## create the mesh for height and altura to visualize. This is know done different because
#  I need to create features so that I can used matrix multiplication
np.random.seed(1)
fig, ax1 = plt.subplots(1,1, figsize = (10,5))

# domain over where we want to plot the function implemented by the NNet.
# since input is 2D we need a mesh
N_points_domain = cg.N_domain_x
thr_prob = cg.thr # use to plot our classification guess
x1, x2 = np.meshgrid(np.linspace(cg.data_x1_lim_l,cg.data_x1_lim_u,N_points_domain),np.linspace(cg.data_x2_lim_l,cg.data_x2_lim_u,N_points_domain))

# reshape for neural network
x_range = np.hstack((np.reshape(x1, (N_points_domain**2,1)),np.reshape(x2, (N_points_domain**2,1))))

# allocate memory to plot decision thresholds
y_range_plot = np.zeros((N_points_domain,N_points_domain), np.float32)


num_models_to_show = cg.N_models_display
bce_loss_old = []
brier_loss_old = []
for i in range(num_models_to_show):
    ## create a candidate model
    w,b = create_computation_graph_linear(n_in = 2,n_out = 1)
    
    ## Take predictions over the grid
    y_range = computation_graph_sigmoid(x_range,w,b)
    
    # reshape back to plotting
    y_range = np.reshape(y_range, (N_points_domain,N_points_domain))

    # start plotting
    ax1.cla()

    # plot dataset    
    ax1.plot(x_data[idx_class0][:,0],x_data[idx_class0][:,1],'o', color = cg.color_class0, markersize = 8, label = f'observations class 0 {cg.name_class0}')
    ax1.plot(x_data[idx_class1][:,0],x_data[idx_class1][:,1],'*', color = cg.color_class1,markersize = 8, label = f'data observations class 1 {cg.name_class1}')
    ax1.set_xlabel(cg.data_x1_name)
    ax1.set_ylabel(cg.data_x2_name)
    
    ## plot prediction probability for class 1 and 0
    idx_range1 = y_range > thr_prob
    idx_range0 = ~idx_range1
    
    y_range_plot[idx_range1] = y_range[idx_range1]
    y_range_plot[idx_range0] = np.nan

    ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Oranges"), alpha = 0.5)
    contourf1 = ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Oranges"), alpha = 0.5)
    
    y_range_plot[idx_range0] = 1-y_range[idx_range0]
    y_range_plot[idx_range1] = np.nan

    ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Blues"), alpha = 0.5)
    contourf2 = ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Blues"), alpha = 0.5)
    
    # decision threshold
    contour1 = ax1.contour(x1, x2, y_range, levels = [thr_prob], colors = ["k"])
    ax1.clabel(contour1, inline=True, fontsize=8, fmt="%.2f")
    
    ## set legend
    ax1.legend()

    ## set contour bar level
    if i == 0:
        cbar2 = fig.colorbar(contourf2, ax=ax1, orientation='vertical')
        cbar2.set_label(f'Probability of being a {cg.name_class0}')
    
        #cbar1 = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # Ajusta la posición [izq, abajo, ancho, alto]
        cbar1 = fig.colorbar(contourf1, ax=ax1, orientation='vertical')
        cbar1.set_label(f'Probability of being a {cg.name_class1}')
    
    ## Cortesía de chatGPT (desde linea siguiente hasta el final de esta celda):
    ## save images for later display
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)  

writer.close() 
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

As always, each of the models has an associated loss. Here, we cannot plot the loss landscape anymore since we have three parameters. We could by freezing, for instancec, $b$. So in this case we show losses associated to each model differently. You can see how models that classify worse the data has higher loss as expected because we are assigning probabilities that differ with the associated labels $t$.

In [ ]:
# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

## number of losses to consider
num_losses = len(cg.losses)

## create the mesh for height and altura to visualize. This is know done different because
#  I need to create features so that I can used matrix multiplication
np.random.seed(cg.seed)
fig, axes = plt.subplots(2, num_losses, figsize = (10,5))
for _a in axes[0]:
    fig.delaxes(_a)
ax1 = fig.add_subplot(2, 1, 1)  # single wide plot on top, regardless of num_losses
ax_list = axes[1]
plt.subplots_adjust(hspace=0.6) 

# domain over where we want to plot the function implemented by the NNet.
# since input is 2D we need a mesh
N_points_domain = cg.N_domain_x
thr_prob = cg.thr # use to plot our classification guess
x1, x2 = np.meshgrid(np.linspace(cg.data_x1_lim_l,cg.data_x1_lim_u,N_points_domain),np.linspace(cg.data_x2_lim_l,cg.data_x2_lim_u,N_points_domain))

# reshape for neural network
x_range = np.hstack((np.reshape(x1, (N_points_domain**2,1)),np.reshape(x2, (N_points_domain**2,1))))

# allocate memory to plot decision thresholds
y_range_plot = np.zeros((N_points_domain,N_points_domain), np.float32)

# to save individual losses, expected losses and parameters used
expected_loss_acc = [0 for i in range(num_losses)]
expected_loss_acc_old = [[] for i in range(num_losses)]

num_models_to_show = cg.N_models_display

for i in range(num_models_to_show):
    
    ## create a candidate model
    w,b = create_computation_graph_linear(n_in = 2,n_out = 1)
    
    ## Take predictions over the grid
    y_range = computation_graph_sigmoid(x_range,w,b)
    
    # reshape back to plotting
    y_range = np.reshape(y_range, (N_points_domain,N_points_domain))

    ## Take predictions over the data
    y_data = computation_graph_sigmoid(x_data,w,b)
    
    # for each of the losses:
    for loss_itet in range(num_losses):
        
        # compute the loss at the predictions
        loss = cg.losses[loss_itet].loss_fun(t_data, y_data)
    
        # accumulate the loss and save both individual and accumulated losses
        expected_loss_acc[loss_itet] = np.sum(loss)
    
    # start plotting
    ax1.cla()
    for _ax in ax_list:
        _ax.cla()
    
    # plot dataset    
    ax1.plot(x_data[idx_class0][:,0],x_data[idx_class0][:,1],'o', color = color_c0, markersize = 8, label = r'observations class 0 cat')
    ax1.plot(x_data[idx_class1][:,0],x_data[idx_class1][:,1],'*', color = color_c1,markersize = 8, label = r'data observations class 1dog')
    ax1.set_xlabel(cg.data_x1_name)
    ax1.set_ylabel(cg.data_x2_name)
    
    ## plot associated loss as title
    title_name = ""
    for loss_itet in range(num_losses):
        title_name += f"{cg.losses[loss_itet].loss_name} {float(expected_loss_acc[loss_itet]):.2f}"
        if loss_itet != num_losses - 1:
            title_name += "\n"
    ax1.set_title(title_name)    

    ## plot prediction probability for class 1 and 0
    idx_range1 = y_range > thr_prob
    idx_range0 = ~idx_range1
    
    y_range_plot[idx_range1] = y_range[idx_range1]
    y_range_plot[idx_range0] = np.nan

    ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Oranges"), alpha = 0.5)
    contourf1 = ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Oranges"), alpha = 0.5)
    
    y_range_plot[idx_range0] = 1-y_range[idx_range0]
    y_range_plot[idx_range1] = np.nan

    ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Blues"), alpha = 0.5)
    contourf2 = ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Blues"), alpha = 0.5)
    
    # decision threshold
    contour1 = ax1.contour(x1, x2, y_range, levels = [thr_prob], colors = ["k"])
    ax1.clabel(contour1, inline=True, fontsize=8, fmt="%.2f")
    
    ## set legend
    ax1.legend()

    ## set contour bar level
    if i == 0:
        cbar2 = fig.colorbar(contourf2, ax=ax1, orientation='vertical')
        cbar2.set_label(f'Probability of being a {cg.name_class0}')
    
        #cbar1 = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # Ajusta la posición [izq, abajo, ancho, alto]
        cbar1 = fig.colorbar(contourf1, ax=ax1, orientation='vertical')
        cbar1.set_label(f'Probability of being a {cg.name_class1}')
    
    ## ===================
    ## Plot loss over ax 2
    for loss_itet, _ax in zip(range(num_losses),ax_list):
        _ax.plot(np.arange(len(expected_loss_acc_old[loss_itet]))+1, expected_loss_acc_old[loss_itet], 'o', color = 'k')
        _ax.plot(i+1, expected_loss_acc[loss_itet], 'o', color = 'k')
        _ax.set_ylabel(f'loss')
        _ax.set_xlabel(f'i-th model')
        _ax.set_title(f'{cg.losses[loss_itet].loss_name} Loss')
        _ax.set_xlim([0 , num_models_to_show+1])
        _ax.set_ylim([0, cg.losses[loss_itet].loss_y_lim_u])
            
    ## =============
    ## Append losses
    for loss_itet in range(num_losses):
         expected_loss_acc_old[loss_itet].append(expected_loss_acc[loss_itet])
    
    ## Cortesía de chatGPT (desde linea siguiente hasta el final de esta celda):
    ## save images for later display
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)  

writer.close() 
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

### Optimization through gradient descent

Again, we can use gradient descent to get the best model. With the BCE loss, we have a convex loss, so there is a unique minimum. Since the data is linearly separable, we can separate our data perfectly.

In [ ]:
get_video = True

if get_video:
    # Create temporary file for video creation
    video_filename = "/tmp/aux.mp4"

    ## video writer
    writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")


## create the mesh for height and altura to visualize. This is know done different because
#  I need to create features so that I can used matrix multiplication
loss_type = 'bce'
np.random.seed(cg.seed)
fig, (ax1,ax2) = plt.subplots(1,2,figsize = (10,5))
plt.subplots_adjust(wspace=0.2) 

# domain over where we want to plot the function implemented by the NNet.
# since input is 2D we need a mesh
N_points_domain = cg.N_domain_x
thr_prob = cg.thr # use to plot our classification guess
x1, x2 = np.meshgrid(np.linspace(cg.data_x1_lim_l,cg.data_x1_lim_u,N_points_domain),np.linspace(cg.data_x2_lim_l,cg.data_x2_lim_u,N_points_domain))

# reshape for neural network
x_range = np.hstack((np.reshape(x1, (N_points_domain**2,1)),np.reshape(x2, (N_points_domain**2,1))))

# allocate memory to plot decision thresholds
y_range_plot = np.zeros((N_points_domain,N_points_domain), np.float32)

## Run gradient descent
lr = 0.1
epochs = 200

## Initialize model
w,b = create_computation_graph_linear(n_in = 2,n_out = 1)
loss_over_training = []
for e in range(epochs):

    ## forward plus backward
    grad_w, grad_b = cg.losses_named[loss_type].grad_loss_fun(x_data,t_data, w, b)
    
    ## Compute loss at current parameter value
    y_data = computation_graph_sigmoid(x_data,w,b)
    
    loss = cg.losses_named[loss_type].loss_fun(t_data, y_data)
        
    loss_acc = np.sum(loss)
        
    loss_over_training.append(loss_acc)
    
    ## =============
    ## Append losses
    loss_over_training.append(loss_acc)
   
    # start plotting
    if get_video:
        ## Take predictions over the grid
        y_range = computation_graph_sigmoid(x_range,w,b)
    
        # reshape back to plotting
        y_range = np.reshape(y_range, (N_points_domain,N_points_domain))
    
        ax1.cla()
        ax2.cla()

        # plot dataset    
        ax1.plot(x_data[idx_class0][:,0],x_data[idx_class0][:,1],'o', color = color_c0, markersize = 8, label = r'observations class 0 cat')
        ax1.plot(x_data[idx_class1][:,0],x_data[idx_class1][:,1],'*', color = color_c1,markersize = 8, label = r'data observations class 1dog')
        ax1.set_xlabel(cg.data_x1_name)
        ax1.set_ylabel(cg.data_x2_name)

        ## plot prediction probability for class 1 and 0
        idx_range1 = y_range > thr_prob
        idx_range0 = ~idx_range1

        y_range_plot[idx_range1] = y_range[idx_range1]
        y_range_plot[idx_range0] = np.nan

        ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Oranges"), alpha = 0.5)
        contourf1 = ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Oranges"), alpha = 0.5)

        y_range_plot[idx_range0] = 1-y_range[idx_range0]
        y_range_plot[idx_range1] = np.nan

        ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Blues"), alpha = 0.5)
        contourf2 = ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Blues"), alpha = 0.5)

        # decision threshold
        contour1 = ax1.contour(x1, x2, y_range, levels = [thr_prob], colors = ["k"])
        ax1.clabel(contour1, inline=True, fontsize=8, fmt="%.2f")

        ## set legend
        ax1.legend()

        ## set contour bar level
        if e == 0:
            cbar2 = fig.colorbar(contourf2, ax=ax1, orientation='vertical')
            cbar2.set_label(f'Probability of being a {cg.name_class0}')

            #cbar1 = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # Ajusta la posición [izq, abajo, ancho, alto]
            cbar1 = fig.colorbar(contourf1, ax=ax1, orientation='vertical')
            cbar1.set_label(f'Probability of being a {cg.name_class1}')

        ## ===================
        ## Plot loss over ax 2
        ax2.plot(np.arange(len(loss_over_training)),loss_over_training, color = 'k')

        ax2.set_ylabel(f'loss')

        ax2.set_xlabel(f'epochs over training')

        ax2.set_title(f'{cg.losses_named[loss_type].loss_name} Loss')

        ax2.set_xlim([0 , epochs])

        
        ## Cortesía de chatGPT (desde linea siguiente hasta el final de esta celda):
        ## save images for later display
        buf = BytesIO()
        fig.savefig(buf, format="png", dpi=100)
    
        buf.seek(0)
        frame = imageio.imread(buf) 
        writer.append_data(frame) 
    
    ## update parameter
    w = w - lr*grad_w
    b = b - lr*grad_b

if get_video:
    writer.close() 

ax2.plot(np.arange(len(loss_over_training)),loss_over_training, color = 'k')

ax2.set_ylabel(f'loss')

ax2.set_xlabel(f'epochs over training')

ax2.set_title(f'{cg.losses_named[loss_type].loss_name} Loss')

ax2.set_xlim([0 , epochs])

print(f"Obtained optimal parameters w={w} b = {b} with associated loss {loss_acc}")

plt.close("all")

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

### The decision boundary

The decision boundary is obtained as before. For a probability threshold, we have the corresponding logit $a$. Then we need to solve $a=\xvect\wvec$. The decision rule remains "decide class $t=1$ if $\xvect\wvec\geq a$. We might also find (and this is also valid for our previous case) rules such as "decide class $t=1$ if $\xvect\wvec-a\geq 0$.

Note that, for $D$ dimensions (in this case we have $D=2$), this results in a single equation with $D$ unknowns. This results in an underdetermined system of equations (https://en.wikipedia.org/wiki/Underdetermined_system) which might have infinite solutions. These infinitely many solutions span a hyperplane of dimension $D-1$. In this case, since D=2 we have that, the decision boundary is a line.

In practice, we do not need any general machinery to write this down: we can simply isolate one coordinate in terms of the rest. From $w_1x_1+w_2x_2+b=a$ we get

$$
x_2=\frac{a-b-w_1x_1}{w_2}
$$

which already parametrizes the whole line. For $D$ dimensions in general, isolating one coordinate in terms of the other $D-1$ works exactly the same way, giving the hyperplane directly.

A numerical way of doing this is to evaluate $z$ on a grid of points and find where it crosses $a$ (i.e., where the sign of $z-a$ changes between neighboring points), interpolating between them -- this is what `contour` plotting routines do internally. This is useful when no analytical solution is possible.


In [ ]:
# decision boundary: w1*x1 + w2*x2 + b = 0  =>  x2 = -(b + w1*x1)/w2
x1_line = np.linspace(cg.data_x1_lim_l, cg.data_x1_lim_u, 200)
x2_line = -(b.item() + w[0, 0]*x1_line) / w[1, 0]

# evaluate the predicted probability on a grid to shade each decided region
x1_grid, x2_grid = np.meshgrid(
    np.linspace(cg.data_x1_lim_l, cg.data_x1_lim_u, 300),
    np.linspace(cg.data_x2_lim_l, cg.data_x2_lim_u, 300),
)
X_grid = np.stack([x1_grid.ravel(), x2_grid.ravel()], axis=1)
z_grid = computation_graph_linear(X_grid, w, b).reshape(x1_grid.shape)
decide_1 = z_grid >= 0

fig, ax = plt.subplots(1, 1, figsize=(7, 6))

ax.contourf(x1_grid, x2_grid, decide_1, levels=[-0.5, 0.5, 1.5], colors=['blue', 'orange'], alpha=0.8)
ax.plot(x1_line, x2_line, color='k', linewidth=3, label='decision boundary')

ax.plot(x_data[idx_class0, 0], x_data[idx_class0, 1], 'o', color=color_c0, markersize=8, label='class 0 (cat)')
ax.plot(x_data[idx_class1, 0], x_data[idx_class1, 1], '*', color=color_c1, markersize=10, label='class 1 (dog)')

ax.set_xlabel(cg.data_x1_name)
ax.set_ylabel(cg.data_x2_name)
ax.set_xlim([cg.data_x1_lim_l, cg.data_x1_lim_u])
ax.set_ylim([cg.data_x2_lim_l, cg.data_x2_lim_u])
ax.legend(loc='upper left')

#### Non-Linear decision boundaries.

As with the one-dimensional case, we might have non-linear decision boundaries. In fact, decision boundaries might arise from different perspectives. For instance, Duda and Hart's book (Pattern Classification) shows how decision thresholds naturally arise from generative classifiers made up from Gaussian distributions. Depending on whether the mean/covariances are diagonal/shared, etc different functional types of decision boundaries arise (quadratic, linear, etc).

One "cool" thing about decision thresholds is that we can identify them and associate to geometrical shapes. For instance, consider a linear basis function model in $2$ dimensions where only quadratic terms appear. The decision threshold might be something like:

$$
\begin{split}
a &= \xvect\wvec\\
a &= w_1x_1^2 + w_2x_2^2
\end{split}
$$

This can be associated with a circle, an ellipse, or a hyperbola, depending on the relation between $w_1$ and $w_2$: a circle in the special case $w_1=w_2$ (same sign), an ellipse when $w_1\neq w_2$ (same sign), and a hyperbola when they have opposite signs.

Note, however, that pure quadratic terms only cover conics centered at the origin. The canonical equation of a circle centered at $(h,k)$ with radius $r$ is $(x_1-h)^2+(x_2-k)^2=r^2$, which expands to

$$
x_1^2+x_2^2-2hx_1-2kx_2+(h^2+k^2-r^2)=0.
$$

So a general circle (or an off-center ellipse/hyperbola) needs the linear terms $x_1,x_2$ and a constant term as well, not just the pure quadratic ones: $\phi(\xvec)=[x_1,x_2,x_1^2,x_2^2]$.

For now, let's focus on the case where the circle is centered at the origin, so that only the quadratic terms matter. Let's build a circular dataset and check that this model can indeed learn to separate it, even though the raw features $x_1,x_2$ alone are not linearly separable.


##### Dataset

In [ ]:
# two concentric noisy circles, centered at the origin
x_data, t_data = make_circles(n_samples=50, noise=0.1, factor=0.1, random_state=cg.seed)
t_data = t_data.reshape(-1, 1)

idx_class0 = (t_data == 0).ravel()
idx_class1 = (t_data == 1).ravel()

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.plot(x_data[idx_class0, 0], x_data[idx_class0, 1], 'o', color=color_c0, markersize=5, label='class 0 (outer ring)')
ax.plot(x_data[idx_class1, 0], x_data[idx_class1, 1], '*', color=color_c1, markersize=7, label='class 1 (inner ring)')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_aspect('equal')
ax.legend()

##### Model optimization

In [ ]:
# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=5, codec="libx264")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.subplots_adjust(wspace=0.3)

margin = 0.5
x1_grid, x2_grid = np.meshgrid(
    np.linspace(x_data[:, 0].min() - margin, x_data[:, 0].max() + margin, 150),
    np.linspace(x_data[:, 1].min() - margin, x_data[:, 1].max() + margin, 150),
)

# quadratic-only features: phi(x) = [x1^2, x2^2], built with generate_features per input dimension
phi1 = generate_features(x_data[:, 0:1], poly_degree=2, add_bias=False)   # [x1, x1^2]
phi2 = generate_features(x_data[:, 1:2], poly_degree=2, add_bias=False)   # [x2, x2^2]
X_feat = np.column_stack([phi1[:, 1], phi2[:, 1]])                       # keep only [x1^2, x2^2]

np.random.seed(cg.seed)
w = np.random.normal(0, 0.1, size=(2, 1))
b = np.array([[0.0]])

lr = 0.25
epochs = 200
frame_every = 1   # one frame every 30 epochs, to keep the video short

loss_history = []
for e in range(epochs):
    y = computation_graph_sigmoid(X_feat, w, b)
    loss_history.append(np.sum(bce_loss_function(t_data, y, clip=1e12)))

    grad_w, grad_b = grad_bce_loss_wrt_sigmoid_model(X_feat, t_data, w, b)

    if e % frame_every == 0 or e == epochs - 1:
        X_feat_grid = np.stack([x1_grid.ravel()**2, x2_grid.ravel()**2], axis=1)
        z_grid = computation_graph_linear(X_feat_grid, w, b).reshape(x1_grid.shape)
        decide_1 = z_grid >= 0

        ax1.cla()
        ax2.cla()

        ax1.contourf(x1_grid, x2_grid, decide_1, levels=[-0.5, 0.5, 1.5], colors=['blue', 'orange'], alpha=0.5)
        ax1.contour(x1_grid, x2_grid, z_grid, levels=[0], colors='k', linewidths=3)
        ax1.plot(x_data[idx_class0, 0], x_data[idx_class0, 1], 'o', color=color_c0, markersize=5, label='class 0 (outer ring)')
        ax1.plot(x_data[idx_class1, 0], x_data[idx_class1, 1], '*', color=color_c1, markersize=7, label='class 1 (inner ring)')
        ax1.set_xlabel('$x_1$')
        ax1.set_ylabel('$x_2$')
        ax1.set_aspect('equal')
        ax1.set_title(f'iteration {e}')
        ax1.legend(loc='upper left')

        ax2.plot(loss_history, color='C0')
        ax2.set_xlim([0, epochs])
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.set_title('BCE loss evolution')

        buf = BytesIO()
        fig.savefig(buf, format="png", dpi=100)
        buf.seek(0)
        frame = imageio.imread(buf)
        writer.append_data(frame)

    w = w - lr * grad_w
    b = b - lr * grad_b

writer.close()
plt.close()

print(f"w={w.ravel()}, b={b.item():.4f}, final BCE loss={loss_history[-1]:.4f}")

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

##### Decision boundary learnt by the model

In [ ]:
# decision boundary: w1*x1^2 + w2*x2^2 + b = 0
margin = 0.5
x1_grid, x2_grid = np.meshgrid(
    np.linspace(x_data[:, 0].min() - margin, x_data[:, 0].max() + margin, 300),
    np.linspace(x_data[:, 1].min() - margin, x_data[:, 1].max() + margin, 300),
)
X_feat_grid = np.stack([x1_grid.ravel()**2, x2_grid.ravel()**2], axis=1)
z_grid = computation_graph_linear(X_feat_grid, w, b).reshape(x1_grid.shape)
decide_1 = z_grid >= 0

fig, ax = plt.subplots(1, 1, figsize=(6, 6))

ax.contourf(x1_grid, x2_grid, decide_1, levels=[-0.5, 0.5, 1.5], colors=['blue', 'orange'], alpha=0.5)
ax.contour(x1_grid, x2_grid, z_grid, levels=[0], colors='k', linewidths=3)

ax.plot(x_data[idx_class0, 0], x_data[idx_class0, 1], 'o', color=color_c0, markersize=5, label='class 0 (outer ring)')
ax.plot(x_data[idx_class1, 0], x_data[idx_class1, 1], '*', color=color_c1, markersize=7, label='class 1 (inner ring)')

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_aspect('equal')
ax.legend(loc='upper left')

## Delving a bit deeper into basis functions.

One cool thing about the circle example is that it allows us to easily gain intuition about what a linear basis function model does. Note that, clearly, a linear basis function model can learn non-linear stuff through a linear relationship $\xvec^t\wvec$ in the parameters.

The question is, what is happening here?. Clearly, $\xvec^t\wvec$ is a linear product, so in some sense, or somewhere, it must be learning something linear. In fact, at the very core of a linear basis function, the idea relies on mapping the input $x$ to a space where things are linearly related, and learning a linear model in that space. In our circle example, $x_1$ and $x_2$, has clearly a non linear shape. However, in the space where the function $x^2$ maps to is actually linear. Let's visualize this:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(x_data[idx_class0, 0], x_data[idx_class0, 1], 'o', color=color_c0, markersize=5, label='class 0 (outer ring)')
ax1.plot(x_data[idx_class1, 0], x_data[idx_class1, 1], '*', color=color_c1, markersize=7, label='class 1 (inner ring)')
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')
ax1.set_aspect('equal')
ax1.set_title('Original input space')
ax1.legend(loc='upper left')

ax2.plot(x_data[idx_class0, 0]**2, x_data[idx_class0, 1]**2, 'o', color=color_c0, markersize=5, label='class 0 (outer ring)')
ax2.plot(x_data[idx_class1, 0]**2, x_data[idx_class1, 1]**2, '*', color=color_c1, markersize=7, label='class 1 (inner ring)')
ax2.set_xlabel('$x_1^2$')
ax2.set_ylabel('$x_2^2$')
ax2.set_title('Feature space $\\phi(x)=[x_1^2,x_2^2]$')
ax2.legend(loc='upper left')

plt.tight_layout()

## Measuring performance

Investigate yourself performance metrics such as accuracy, f1 score, precision, recall, auc curve etc.

## TODO
* Newton method for logistic regressionw which seems to be something similar to EM in student-t, i.e.  a weighted model.
* Evaluation metrics beyond the loss itself: accuracy, confusion matrix, ROC/AUC.
* Alternative links for the same Bernoulli parameter: probit, cloglog (both already introduced in the [GLM chapter](4_Generalized_Linear_Models_theory.ipynb)), and how they change the decision boundary/robustness trade-off.
* MAP / Bayesian logistic regression (eg the Laplace approximation).
* General exponential-family derivation of "canonical link", beyond the Bernoulli case worked out here.
* Robust classification
